# 04 · Búsqueda exploratoria original

Cuatro recetas, diez épocas como máximo y semilla 11. Exploratoria: no sustituye los resultados de la receta principal.

Ejecutar todas las celdas con GPU. Requiere paquete_colab_e2_v2.zip. Salida nueva: `/content/drive/MyDrive/tesis_imo_e3_secuencia_v4/hpo`. Repetir reanuda; no mezclar con el piloto v3. No se evalúan imágenes de test.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import subprocess, sys, os, json, hashlib, zipfile
subprocess.run(['nvidia-smi'], check=True)
subprocess.run(['mountpoint', '-q', '/content/drive'], check=True)
OUTPUT = Path('/content/drive/MyDrive/tesis_imo_e3_secuencia_v4/hpo')
OUTPUT.mkdir(parents=True, exist_ok=True)


In [ ]:
EXPECTED_ZIP_SHA256 = '954f4fe5663d9fedd676dcee4f1f7fb9086a2a7f5c1907ce902b5df4c109cb49'
ZIP = next((p for p in [Path('/content/paquete_colab_e2_v2.zip'), Path('/content/drive/MyDrive/paquete_colab_e2_v2.zip')] if p.exists()), None)
if ZIP is None: raise FileNotFoundError('Sube paquete_colab_e2_v2.zip al panel de archivos y repite esta celda.')
def sha_file(p):
    h=hashlib.sha256()
    with p.open('rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''): h.update(b)
    return h.hexdigest()
assert sha_file(ZIP)==EXPECTED_ZIP_SHA256, 'ZIP incorrecto o modificado; usa el archivo v2 entregado con este notebook'
BASE=Path('/content/imo_e2_v2')
BASE.mkdir(exist_ok=True)
with zipfile.ZipFile(ZIP) as z:
    for n in z.namelist():
        if not (BASE/n).resolve().is_relative_to(BASE): raise ValueError('Ruta no válida en ZIP')
    z.extractall(BASE)
manifest=json.loads((BASE/'PACKAGE_MANIFEST.json').read_text())
for name, expected in manifest['files'].items():
    assert sha_file(BASE/name)==expected, name
PROJECT=BASE/'tesis_imo_para_codex'
print('Paquete íntegro:',len(manifest['files']),'archivos')


## Entorno aislado y fijado
Se crea Python 3.11 fuera del kernel de Colab. La instalación puede tardar varios minutos; no cambia el Python del notebook. No se compilan extensiones CUDA: se usan ruedas publicadas por sus autores para torch 2.5/CUDA 12.


In [ ]:
subprocess.run([sys.executable,'-m','pip','install','uv==0.6.9'],check=True)
ENV=Path('/content/imo-e2-env')
if not (ENV/'bin/python').exists():
    subprocess.run([sys.executable,'-m','uv','venv','--python','3.11',str(ENV)],check=True)
PY=str(ENV/'bin/python')
def pip_install(*items):
    subprocess.run([sys.executable,'-m','uv','pip','install','--python',PY,*items],check=True)
pip_install('torch==2.5.1','torchvision==0.20.1','--index-url','https://download.pytorch.org/whl/cu121')
pip_install('-r',str(PROJECT/'requirements-e2.txt'))
subprocess.run([PY,'-c','import torch, torchvision; assert torch.cuda.is_available(); print(torch.__version__,torch.cuda.get_device_name(0))'],check=True)
freeze=subprocess.check_output([sys.executable,'-m','uv','pip','freeze','--python',PY],text=True)
(OUTPUT/'environment.freeze.txt').write_text(freeze)


In [ ]:
pip_install('opencv-python-headless==4.11.0.86', 'pytest==8.3.5', 'pyclipper==1.3.0.post6', 'onnx==1.17.0', 'onnxruntime==1.20.1')
subprocess.run([PY, '-c', "import cv2,numpy; assert cv2.__version__=='4.11.0'; assert numpy.__version__=='1.26.4'; print(cv2.__version__,numpy.__version__)"],check=True)


In [ ]:
CONTENIDO = {'src/__init__.py': '', 'src/geometria_at.py': '"""Geometria de DBNet-AT: etiquetas (C1), expansion consistente (C2) y posproceso. Modulo autonomo.\n\nFuentes:\n- DBNet: Liao, Wan, Yao, Chen y Bai (2020), "Real-time Scene Text Detection with Differentiable Binarization",\n  AAAI. De ahi salen el encogimiento D = A(1 - r^2)/L con r = 0,4, el mapa de umbral y la expansion D\' = A\' r\'/L\'\n  con r\' = 1,5. Los valores 0,3 (binarizacion) y 0,5 (puntaje minimo de region) son los del codigo oficial.\n- Formulacion de DBNet-AT: entregable_3/diseno.md, seccion 3 (C1 en 3.2, C2 en 3.3, teorema en 3.5).\n\nNotacion (la del teorema de 3.5):\n- K es una region (poligono convexo si se quiere la garantia exacta). A, L y g son su area, su perimetro y su\n  radio inscrito (la mayor distancia de un pixel de K al fondo).\n- r = 0,4 es la razon de encogimiento de DBNet; g0 = 4 px es el paso del mapa mas profundo de la U-Net compacta\n  (1/4 de 256), fijado por la arquitectura.\n- phi(g) = g^2 / (g^2 + g0^2) es la atenuacion por grosor de C1. Con g0 = 0, phi = 1 y se recupera DBNet.\n- La regla de encogimiento es rho(K) = phi(g) (1 - r^2) A / L. El nucleo es K_{-D} = {x en K : dist(x, borde) >= D}\n  con D = rho(K).\n- El desplazamiento en inglete de un poligono P = interseccion de {n_j . x <= c_j} es\n  M(P, d) = interseccion de {n_j . x <= c_j + d}: se mueve cada lado d hacia afuera y se cortan los consecutivos.\n- C2 busca el d que cumple d = rho(M(K\', d)), con K\' el nucleo predicho. Teorema: si K es convexo, ningun lado\n  colapsa al encogerse (S1) y sus angulos interiores son >= 60 grados (S2), la solucion es unica, d = D, y\n  M(K\', D) = K.\n\nCambio respecto a la version de entregable_3/codigo/etiquetas_at.py: el desplazamiento en inglete ya no lo hace\npyclipper (algoritmo de Vatti) sino el codigo de este archivo (inglete_poligono / rellenar_inglete). pyclipper\nqueda solo como prueba cruzada en tests/test_original_model.py.\n\nTodo lo demas viene de E2: instancias por componente conexo (vecindad 8), perimetro con cv2.arcLength, encoger\ny expandir con la transformada de distancia, umbrales del mapa 0,3-0,7, binarizacion 0,3, puntaje minimo 0,5 y\narea minima 9 px.\n"""\nimport cv2\nimport numpy as np\nfrom scipy import ndimage\n\n# ----------------------------------------------------------------------------- constantes\nG0 = 4.0            # px: paso del mapa mas profundo de la U-Net compacta (e3, 1/4 de 256)\nR = 0.4             # razon de encogimiento de DBNet\nR_PRIMA = 1.5       # razon de expansion fija de DBNet\nLIMITE_INGLETE = 2.0  # limite de inglete (multiplos de d). Una esquina de 90 grados necesita raiz(2)\nESCALA = 16         # las coordenadas del relleno van en 1/16 de pixel (cv2.fillPoly con shift = 4)\nAJUSTE_PX = 0.75    # corrimiento de discretizacion del contorno (calibrar_ajuste.py, 200 imagenes de train)\nOCHO = np.ones((3, 3), bool)\n\n\n# ----------------------------------------------------------------------------- primitivas de E2 (reescritas)\ndef instancias(mascara):\n    """Cada componente conexo (vecindad 8) de la mascara es una instancia. Devuelve (etiquetas, n)."""\n    return ndimage.label(mascara > 0, structure=OCHO)\n\n\ndef area_perimetro(inst):\n    """A = numero de pixeles; L = longitud del contorno externo (cv2.arcLength). L >= 1 para no dividir por 0."""\n    contornos, _ = cv2.findContours(inst.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)\n    perim = sum(cv2.arcLength(c, True) for c in contornos)\n    return float(inst.sum()), max(float(perim), 1.0)\n\n\ndef encoger(inst, d):\n    """Nucleo: los pixeles a distancia > d del fondo (equivale al desplazamiento de Vatti con esquinas vivas\n    en poligonos convexos, Lema 1 de diseno.md 3.5)."""\n    return ndimage.distance_transform_edt(inst) > d\n\n\ndef expandir(inst, d):\n    """Expansion euclidea: se anaden los pixeles de fondo a distancia <= d (esquinas redondeadas, como DBNet)."""\n    return ndimage.distance_transform_edt(~inst) <= d\n\n\ndef componentes(binaria, area_min):\n    """Etiqueta los componentes de una mascara y devuelve (etiquetas, lista de etiquetas con area >= area_min)."""\n    etq, n = instancias(binaria)\n    if n == 0:\n        return etq, []\n    areas = ndimage.sum(np.ones_like(etq), etq, index=np.arange(1, n + 1))\n    return etq, [i + 1 for i, a in enumerate(areas) if a >= area_min]\n\n\n# ----------------------------------------------------------------------------- C1: la regla de encogimiento\ndef phi(g, g0=G0):\n    """Atenuacion por grosor de C1: g^2 / (g^2 + g0^2). Con g0 <= 0 vale 1 (C1 apagado, regla de DBNet)."""\n    return 1.0 if g0 <= 0 else g * g / (g * g + g0 * g0)\n\n\ndef regla(region, r=R, g0=G0):\n    """rho(K) = phi(g) (1 - r^2) A / L: distancia de encogimiento de una region (bool)."""\n    A, L = area_perimetro(region)\n    g = float(ndimage.distance_transform_edt(region).max()) if region.any() else 0.0\n    return phi(g, g0) * A * (1 - r ** 2) / L\n\n\ndef at_objetivos(y, v, r=R, g0=G0, tmin=0.3, tmax=0.7):\n    """Etiquetas de entrenamiento (las de DBNet con D = rho(K) en lugar de A(1 - r^2)/L).\n\n    y: mascara binaria (H, W) de rotulos; v: zona valida (bool). Devuelve, con un eje de canal delante:\n    db_gt (nucleo, el blanco de P y de B), db_mask (donde cuenta la perdida), db_umbral (blanco de T, entre tmin y\n    tmax) y db_umbral_mask (la banda donde cuenta la perdida de T). La banda de T usa el mismo D que el nucleo.\n    Una instancia cuyo nucleo desaparece se ignora (db_mask = 0), como el codigo oficial de DBNet.\n    Con g0 = 0 es exactamente la etiqueta de DBNet."""\n    etq, n = instancias(y)\n    gt = np.zeros(y.shape, np.float32)\n    mask = v.astype(np.float32).copy()\n    umbral = np.zeros(y.shape, np.float32)\n    umbral_mask = np.zeros(y.shape, np.float32)\n    for k in range(1, n + 1):\n        inst = etq == k\n        D = regla(inst, r, g0)\n        s = encoger(inst, D)\n        if not s.any():\n            mask[inst] = 0\n            continue\n        gt[s] = 1\n        gd = expandir(inst, D)\n        dist = np.where(inst, ndimage.distance_transform_edt(inst), ndimage.distance_transform_edt(~inst))\n        t = np.clip(1 - dist / max(D, 1e-6), 0, 1)  # 1 en el borde de la region, 0 a distancia D\n        umbral[gd] = np.maximum(umbral[gd], t[gd])\n        umbral_mask[gd] = 1\n    umbral = umbral * (tmax - tmin) + tmin\n    return {"db_gt": gt[None], "db_mask": mask[None], "db_umbral": umbral[None], "db_umbral_mask": umbral_mask[None]}\n\n\n# ----------------------------------------------------------------------------- inglete propio (reemplaza a pyclipper)\ndef _orientar(p):\n    """Poligono (n, 2) float sin vertices repetidos y con area con signo positiva (orden antihorario en el\n    plano x-y; en coordenadas de imagen, con y hacia abajo, es el horario visual: no importa, la aritmetica es la\n    misma). Con esa orientacion la normal exterior del lado (dx, dy) es (dy, -dx)/|e|."""\n    p = np.asarray(p, float)\n    q = p[np.any(np.abs(p - np.roll(p, 1, axis=0)) > 1e-9, axis=1)]  # quita vertices consecutivos repetidos\n    area2 = np.sum(q[:, 0] * np.roll(q[:, 1], -1) - np.roll(q[:, 0], -1) * q[:, 1])\n    return q if area2 >= 0 else q[::-1]\n\n\ndef _geometria_lados(p):\n    """Vectores unitarios t_i y normales exteriores n_i de los lados i = p[i] -> p[i+1] de un poligono orientado."""\n    e = np.roll(p, -1, axis=0) - p\n    t = e / np.linalg.norm(e, axis=1, keepdims=True)\n    n = np.stack([t[:, 1], -t[:, 0]], axis=1)\n    return t, n\n\n\ndef _esquina(v, t1, n1, t2, n2, d, limite):\n    """Puntos de la esquina desplazada en el vertice v, entre el lado entrante (t1, n1) y el saliente (t2, n2).\n\n    Inglete: la interseccion de las dos rectas desplazadas, m = v + d (n1 + n2) / (1 + n1 . n2). Esta a una\n    distancia d / sen(theta/2) de v (theta = angulo interior). Si esa distancia pasa de limite * d, la punta se\n    corta con una recta perpendicular a la bisectriz, a limite * d de v (Clipper la llama \'squaring\'); entonces\n    salen dos puntos, uno sobre cada recta desplazada. Con limite = 2 el corte aparece bajo 60 grados (supuesto S2\n    del teorema). Devuelve una lista de 1 o 2 puntos."""\n    den = 1.0 + float(n1 @ n2)\n    if den > 1e-12 and 1.0 / np.sqrt(den / 2.0) <= limite:\n        return [v + d * (n1 + n2) / den]\n    u = n1 + n2\n    nu = np.linalg.norm(u)\n    u = u / nu if nu > 1e-12 else t1  # giro de 180 grados: se corta perpendicular al lado entrante\n    L = limite * d\n    pts = []\n    for t, n in ((t1, n1), (t2, n2)):\n        ct = float(t @ u)\n        if abs(ct) < 1e-12:\n            pts.append(v + d * n)\n        else:\n            pts.append(v + d * n + ((L - d * float(n @ u)) / ct) * t)\n    return pts\n\n\ndef inglete_poligono(p, d, limite=LIMITE_INGLETE):\n    """Desplazamiento en inglete M(P, d) de un poligono CONVEXO p (n, 2): cada lado d hacia afuera y se cortan los\n    consecutivos (diseno.md 3.5). Devuelve el poligono desplazado (m, 2); m >= n si hay esquinas cortadas.\n\n    Solo es correcto para poligonos convexos y d >= 0. Con un poligono no convexo, las esquinas entrantes\n    cruzarian los lados: para eso esta rellenar_inglete, que usa piezas convexas."""\n    p = _orientar(p)\n    t, n = _geometria_lados(p)\n    salida = []\n    for i in range(len(p)):\n        j = i - 1  # lado entrante al vertice i\n        salida.extend(_esquina(p[i], t[j], n[j], t[i], n[i], d, limite))\n    return np.asarray(salida)\n\n\ndef _piezas_inglete(p, d, limite=LIMITE_INGLETE):\n    """Piezas convexas cuya union es M(P, d) para CUALQUIER poligono simple (convexo o no): el poligono mismo, un\n    rectangulo sobre cada lado (de p[i], p[i+1] a d hacia afuera) y una cuna en cada vertice convexo (la esquina de\n    _esquina). En un vertice entrante no se pone cuna: los rectangulos de sus dos lados ya se solapan alli.\n    Caso limite documentado: el desplazamiento por union de piezas no recorta entrantes mas cortos que d; es el\n    desplazamiento \'exacto\' del conjunto, no el contorno que daria un recorte de bucles como el de Vatti."""\n    p = _orientar(p)\n    t, n = _geometria_lados(p)\n    k = len(p)\n    piezas = [p]\n    for i in range(k):\n        a, b = p[i], p[(i + 1) % k]\n        piezas.append(np.array([a, b, b + d * n[i], a + d * n[i]]))\n        j = i - 1\n        if t[j][0] * t[i][1] - t[j][1] * t[i][0] > 1e-12:  # vertice convexo (giro hacia el interior)\n            punta = _esquina(p[i], t[j], n[j], t[i], n[i], d, limite)\n            piezas.append(np.array([p[i], p[i] + d * n[j], *punta, p[i] + d * n[i]]))\n    return piezas\n\n\ndef _pintar(salida, poligono):\n    """Rellena un poligono float en la mascara uint8 (coordenadas en 1/16 de pixel, sin suavizado)."""\n    pts = np.round(np.asarray(poligono) * ESCALA).astype(np.int32)\n    cv2.fillPoly(salida, [pts], 1, lineType=cv2.LINE_8, shift=4)\n\n\ndef _contornos(c):\n    """Contornos externos del nucleo c (bool), simplificados con Douglas-Peucker a 1 px para que la escalera de\n    pixeles de un borde inclinado no genere picos al unir en inglete."""\n    cont, _ = cv2.findContours(c.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)\n    return [(cv2.approxPolyDP(k, 1.0, True) if len(k) > 4 else k)[:, 0, :].astype(float) for k in cont]\n\n\ndef rellenar_inglete(forma, contornos, d, limite=LIMITE_INGLETE):\n    """Mascara uint8 (forma) con el desplazamiento en inglete, d hacia afuera, de los contornos dados.\n    Contorno convexo: un solo poligono (inglete_poligono). No convexo: union de piezas (_piezas_inglete)."""\n    salida = np.zeros(forma, np.uint8)\n    for p in contornos:\n        if len(p) < 3:\n            continue\n        if cv2.isContourConvex(np.round(p).astype(np.int32)[:, None, :]):\n            _pintar(salida, inglete_poligono(p, d, limite))\n        else:\n            for pieza in _piezas_inglete(p, d, limite):\n                _pintar(salida, pieza)\n    return salida\n\n\ndef expandir_inglete(c, d, ajuste=None, limite=LIMITE_INGLETE):\n    """Desplaza hacia afuera el contorno del nucleo c en d px con uniones en inglete (esquinas vivas) y rellena.\n\n    El contorno de cv2 pasa por los centros de los pixeles del borde y el relleno no coincide exactamente con el\n    conjunto de pixeles de la etiqueta: se desplaza d + ajuste (AJUSTE_PX, un corrimiento de discretizacion que no\n    depende del tamano y se calibro solo con imagenes de train). Devuelve una mascara bool que incluye a c."""\n    desp = d + (AJUSTE_PX if ajuste is None else ajuste)\n    if desp <= 0:\n        return c.copy()\n    salida = rellenar_inglete(c.shape, _contornos(c), desp, limite)\n    return (salida > 0) | c\n\n\n# ----------------------------------------------------------------------------- C2: expansion consistente\ndef expansion_consistente(comp, v, r=R, g0=G0, iteraciones=16, inglete=True, ajuste=None):\n    """d tal que regla(expandir(comp, d) & v) = d, por biseccion; devuelve la region expandida (bool, tamano\n    completo). Es la inversa de C1: M(K\', d) con d = rho(M(K\', d)) (diseno.md 3.3). La region se recorta a su caja\n    con un margen suficiente para que los calculos sean locales.\n    inglete=True expande con esquinas vivas; False, por distancia euclidea (esquinas redondas)."""\n    ys, xs = np.nonzero(comp)\n    g_k = float(ndimage.distance_transform_edt(comp).max())\n    alto = 6.0 * (g_k + 1.0) + 2.0  # cota: en una franja larga d ~ 5,25 g\'; en una compacta d ~ 0,72 g\'\n    m = int(np.ceil(1.5 * alto)) + 2  # el inglete a 90 grados sale raiz(2) d por la esquina\n    y0, y1 = max(ys.min() - m, 0), min(ys.max() + m + 1, comp.shape[0])\n    x0, x1 = max(xs.min() - m, 0), min(xs.max() + m + 1, comp.shape[1])\n    c, vv = comp[y0:y1, x0:x1], v[y0:y1, x0:x1]\n    fuera = None if inglete else ndimage.distance_transform_edt(~c)\n\n    def expandida(d):\n        return (expandir_inglete(c, d, ajuste) if inglete else fuera <= d) & vv\n\n    def h(d):  # h(d) = rho(M(K\', d)) - d es estrictamente decreciente (Lema 2): una sola raiz\n        return regla(expandida(d), r, g0) - d\n\n    bajo = 0.0\n    if h(alto) > 0:  # no deberia pasar; si pasa, se usa la cota y se sigue\n        d = alto\n    else:\n        for _ in range(iteraciones):\n            medio = (bajo + alto) / 2\n            if h(medio) > 0:\n                bajo = medio\n            else:\n                alto = medio\n        d = (bajo + alto) / 2\n    salida = np.zeros(comp.shape, bool)\n    salida[y0:y1, x0:x1] = _refinar(c, d, expandida, r, g0)\n    return salida\n\n\ndef _refinar(c, d_bis, expandida, r, g0, ventana=(-2.5, 1.5), paso=0.25):\n    """Inversa en pixeles. La biseccion da un d continuo, pero la region expandida cambia a saltos (un anillo de\n    pixeles cada vez) y la region correcta puede quedar en el escalon de al lado del cruce (en un cuadrado de 32 px\n    con C1, la etiqueta usa D = 6,77 y la region original aparece para d en [6,0; 6,75), donde regla - d > 0).\n    Entre las regiones distintas que da d en la ventana [d - 2,5; d + 1,5] (pasos de 0,25) se elige la que,\n    encogida con la regla de la etiqueta, reproduce mejor el nucleo observado (IoU). Empate: la de d mas cercano\n    al cruce de la biseccion."""\n    vistas, mejor = set(), (-1.0, 0.0, None)\n    for d in np.arange(max(d_bis + ventana[0], 0.0), d_bis + ventana[1] + 1e-9, paso):\n        reg = expandida(float(d))\n        clave = (int(reg.sum()), hash(np.packbits(reg).tobytes()))\n        if clave in vistas or not reg.any():\n            continue\n        vistas.add(clave)\n        nuc = encoger(reg, regla(reg, r, g0))\n        inter = int((nuc & c).sum())\n        iou = inter / (int(nuc.sum()) + int(c.sum()) - inter)\n        if iou > mejor[0] + 1e-12 or (abs(iou - mejor[0]) <= 1e-12 and abs(d - d_bis) < mejor[1]):\n            mejor = (iou, abs(d - d_bis), reg)\n    return mejor[2] if mejor[2] is not None else expandida(d_bis)\n\n\n# ----------------------------------------------------------------------------- posproceso\ndef at_post(prob, v, umbral=0.3, puntaje_min=0.5, r=R, g0=G0, consistente=True, inglete=True, r_prima=R_PRIMA,\n            area_min=9, ajuste=None):\n    """Inferencia: binarizar P con 0,3, descartar componentes de area < 9 px o P media < 0,5 (como el codigo\n    oficial de DBNet) y expandir cada nucleo. consistente=True: con el d de C2. consistente=False: con r\' fijo,\n    como DBNet (D\' = A\' r\'/L\'), por distancia euclidea o, con inglete=True, en inglete (ablacion factorial)."""\n    etq, keep = componentes((prob > umbral) & v, area_min)\n    salida = np.zeros(prob.shape, bool)\n    for k in keep:\n        comp = etq == k\n        if prob[comp].mean() < puntaje_min:\n            continue\n        if consistente == "okr":\n            salida |= expandir(comp, distancia_okr(comp))\n        elif consistente:\n            salida |= expansion_consistente(comp, v, r, g0, inglete=inglete, ajuste=ajuste)\n        else:\n            A, L = area_perimetro(comp)\n            salida |= expandir_inglete(comp, A * r_prima / L, ajuste) if inglete else expandir(comp, A * r_prima / L)\n    return salida & v\n\n\ndef distancia_okr(comp):\n    """Distancia de dilatacion de OKR (Ma et al., 2023, ICASSP, doi 10.1109/ICASSP49357.2023.10094734, Ec. 2), el\n    trabajo mas cercano (experiments/novedad/lectura_okr.md): S = (C - sqrt(max(C^2 - 16 A, 0))) / 4, con C y A el\n    perimetro y el area de la region predicha. En un rectangulo a x b, C^2 - 16A = 4 (a - b)^2 y S = min(a, b): el\n    lado corto del nucleo. S es el TAMANO del elemento estructurante de la dilatacion (un diametro), igual que el\n    S de su erosion (Ec. 1, S = min lado / 2), asi que la distancia por lado es S / 2: el nucleo de ancho w/2 vuelve\n    a ancho w. Lectura declarada: el articulo redondea S a impar y no precisa la forma del elemento; aqui se usa\n    S / 2 sin redondeo y la expansion euclidea (expandir), para comparar solo la regla que elige la distancia contra\n    el punto fijo de C2."""\n    A, C = area_perimetro(comp)\n    return (C - np.sqrt(max(C * C - 16.0 * A, 0.0))) / 8.0\n\n\n# Variantes de la ablacion factorial 2^3 (diseno.md 5.1): nombre -> (g0, d consistente, inglete).\n# Ademas, una variante de comparacion con el trabajo mas cercano (protocolo §8): \'okr_sobre_at\' usa los nucleos del\n# brazo dbnet_at (C1) y la dilatacion de OKR en lugar de C2. Es exploratoria, fuera de la familia de la ablacion.\n# \'dbnet\' es el base: regla y posproceso de DBNet (g0 = 0, r\' fijo, expansion euclidea).\nVARIANTES = {\n    "dbnet": (0.0, False, False),\n    "c1_rfijo": (G0, False, False),        # solo encogimiento atenuado, con r\' = 1,5\n    "inglete_rfijo": (0.0, False, True),   # regla de DBNet, r\' = 1,5, expansion en inglete (C2b solo)\n    "c1_inglete_rfijo": (G0, False, True),  # C1 con r\' = 1,5 en inglete (sin C2a)\n    "c2_redondo": (0.0, True, False),      # regla de DBNet, d consistente, expansion euclidea\n    "c2": (0.0, True, True),               # regla de DBNet, d consistente, expansion en inglete\n    "at_redondo": (G0, True, False),       # C1 + d consistente con expansion euclidea\n    "dbnet_at": (G0, True, True),          # el modelo propio: C1 + C2\n    "okr_sobre_at": (G0, "okr", False),    # nucleos C1 + dilatacion de OKR (comparacion con el trabajo mas cercano)\n}\n\n\ndef objetivos_de(variante):\n    """Funcion (y, v) -> objetivos de entrenamiento de la variante. Solo importa g0 (C1)."""\n    g0 = VARIANTES[variante][0]\n    return lambda y, v: at_objetivos(y, v, g0=g0)\n\n\ndef post_de(variante):\n    """Funcion (P, v) -> mascara predicha de la variante (posproceso con o sin C2, con o sin inglete)."""\n    g0, cons, ing = VARIANTES[variante]\n    return lambda p, v: at_post(p, v, g0=g0, consistente=cons, inglete=ing)\n\n\n# DBNet base, con los nombres de E2\ndef db_objetivos(y, v, r=R, tmin=0.3, tmax=0.7):\n    return at_objetivos(y, v, r=r, g0=0.0, tmin=tmin, tmax=tmax)\n\n\ndef db_post(prob, v, **kw):\n    return at_post(prob, v, g0=0.0, consistente=False, inglete=False, **kw)\n', 'src/original_model.py': '"""UNetAT: la U-Net compacta del benchmark E2 con el mecanismo de DBNet-AT. Modelo original del Entregable 3.\n\nQue es. Se porta el mecanismo de DBNet (Liao et al., 2020, AAAI, "Real-time Scene Text Detection with\nDifferentiable Binarization") y el de DBNet-AT (entregable_3/diseno.md, seccion 3) a la U-Net compacta de E2\n(modelos_base.SmallUNet: tres niveles de 16, 32 y 64 canales, sin BatchNorm). Se entrena desde cero, sin pesos\npreentrenados, en el mismo pipeline de E2.\n\nArquitectura (la de SmallUNet, reescrita aqui, mas dos cabezas):\n    e1 = bloque(3 -> 16)             resolucion completa\n    e2 = bloque(16 -> 32)            1/2\n    e3 = bloque(32 -> 64)            1/4   (el mapa mas profundo: su paso, 4 px, es g0 de C1)\n    d2 = bloque(96 -> 32)            sube e3 y concatena e2\n    d1 = bloque(48 -> 16)            sube d2 y concatena e1\n    cabeza_prob, cabeza_umbral: dos cabezas sobre los 16 canales de d1 que dan los logits de P y de T.\nbloque(a, b) = Conv3x3(a,b) ReLU Conv3x3(b,b) ReLU. Las capas del codificador y del decodificador se crean en\nel mismo orden que en SmallUNet, asi que con la misma semilla global tienen los mismos pesos iniciales que la\nU-Net de E2; las cabezas se crean al final.\n\nFormulacion (DBNet, seccion 3 del articulo):\n    P = sigmoid(z_P)                       mapa de probabilidad del nucleo\n    T = sigmoid(z_T)                       mapa de umbral aprendido\n    B = sigmoid(k (P - T)),  k = 50        binarizacion diferenciable (aproxima el escalon H(P - T))\nDerivada de la binarizacion, la que hace entrenable al umbral:\n    dB/dP =  k B (1 - B)\n    dB/dT = -k B (1 - B)\nLa derivada es grande solo donde B ~ 0,5, es decir P ~ T: ahi el gradiente de la perdida empuja a P y a T a\nsepararse. Lejos del cruce B esta saturado y casi no hay gradiente.\n\nPerdida (perdida_db), la del codigo oficial de DBNet y la de E2 (entregable_2/codigo/modelos.py, DBNet.perdida):\n    L = 5 Ls + 10 Lt + Lb\n    Ls = BCE de P con minado de negativos dificiles 1:3 (OHEM)\n    Lt = L1 entre T y su blanco, promediada dentro de la banda Gd del mapa de umbral\n    Lb = Dice entre B y el nucleo\nLos pesos 5, 10 y 1 coinciden con diseno.md 3.1 ("5 Ls + 10 Lt + Lb"). El articulo escribe BCE para Lb con alfa = 1;\nel codigo oficial usa Dice y pesa 5 la BCE, y E2 sigue el codigo.\n\nLas etiquetas (nucleo, banda y blanco de T) las produce src.geometria_at.at_objetivos (C1 si variante=\'dbnet_at\',\nregla de DBNet si variante=\'dbnet\'). El posproceso (binarizar P, filtrar, expandir con C2) es predict_mask.\n"""\nfrom contextlib import nullcontext\n\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\nfrom . import geometria_at as G\n\nK_BINARIZACION = 50.0\n\n\ndef bloque(a, b):\n    """Dos convoluciones 3x3 con ReLU (el bloque de SmallUNet)."""\n    return nn.Sequential(nn.Conv2d(a, b, 3, padding=1), nn.ReLU(), nn.Conv2d(b, b, 3, padding=1), nn.ReLU())\n\n\nclass UNetAT(nn.Module):\n    """U-Net compacta con cabezas de probabilidad (P) y de umbral (T) y binarizacion B = sigmoid(k (P - T)).\n\n    Parametros:\n        k: pendiente de la binarizacion aproximada (50, como DBNet).\n        cabeza: \'1x1\' (una convolucion 1x1 por cabeza: 17 parametros cada una, paridad de capacidad con la U-Net\n            de E2) o \'3x3\' (conv 3x3 de 16 canales, ReLU y conv 1x1: mas capacidad en cada cabeza).\n        semilla: si se da, los pesos iniciales salen de esa semilla sin tocar el generador global; si es None se\n            usa el generador global (el arnes de E2 lo siembra con setup_seed antes de crear el modelo).\n    forward: en entrenamiento devuelve un dict con \'prob\' (P), \'thresh\' (T), \'binary\' (B) y \'logit_prob\' (z_P, que\n        la perdida necesita para la BCE estable); en eval devuelve solo P, de forma (B, 1, H, W).\n    """\n\n    def __init__(self, k=K_BINARIZACION, cabeza="1x1", semilla=None):\n        super().__init__()\n        if cabeza not in ("1x1", "3x3"):\n            raise ValueError("cabeza debe ser \'1x1\' o \'3x3\'")\n        self.k = float(k)\n        self.cabeza_tipo = cabeza\n        # fork_rng(devices=[]) guarda y restaura el generador de la CPU: crear el modelo con semilla no altera el global\n        contexto = torch.random.fork_rng(devices=[]) if semilla is not None else nullcontext()\n        with contexto:\n            if semilla is not None:\n                torch.manual_seed(int(semilla))\n            # mismo orden de creacion que SmallUNet: e1, e2, e3, d2, d1\n            self.e1, self.e2, self.e3 = bloque(3, 16), bloque(16, 32), bloque(32, 64)\n            self.d2, self.d1 = bloque(96, 32), bloque(48, 16)\n            self.cabeza_prob = self._cabeza(cabeza)\n            self.cabeza_umbral = self._cabeza(cabeza)\n\n    @staticmethod\n    def _cabeza(tipo):\n        if tipo == "1x1":\n            return nn.Conv2d(16, 1, 1)\n        return nn.Sequential(nn.Conv2d(16, 16, 3, padding=1), nn.ReLU(), nn.Conv2d(16, 1, 1))\n\n    # ------------------------------------------------------------------ validacion de la entrada\n    @staticmethod\n    def _validar(x):\n        if not isinstance(x, torch.Tensor):\n            raise ValueError(f"la entrada debe ser un tensor, no {type(x).__name__}")\n        if x.dim() != 4:\n            raise ValueError(f"se espera (B, 3, H, W); llego {tuple(x.shape)}")\n        if x.shape[1] != 3:\n            raise ValueError(f"se esperan 3 canales; llegaron {x.shape[1]}")\n        if not x.is_floating_point():\n            raise ValueError(f"se espera un tensor flotante, no {x.dtype}")\n        h, w = x.shape[-2:]\n        if h % 4 or w % 4:\n            raise ValueError(f"alto y ancho deben ser multiplos de 4 (dos reducciones a la mitad); llego {h} x {w}")\n\n    # ------------------------------------------------------------------ red\n    def _caracteristicas(self, x):\n        a = self.e1(x)\n        b = self.e2(F.max_pool2d(a, 2))\n        c = self.e3(F.max_pool2d(b, 2))\n        d = self.d2(torch.cat([F.interpolate(c, size=b.shape[-2:], mode="nearest"), b], 1))\n        return self.d1(torch.cat([F.interpolate(d, size=a.shape[-2:], mode="nearest"), a], 1))\n\n    def forward(self, x):\n        self._validar(x)\n        h = self._caracteristicas(x)\n        zp = self.cabeza_prob(h)\n        if not self.training:\n            return torch.sigmoid(zp)\n        p, t = torch.sigmoid(zp), torch.sigmoid(self.cabeza_umbral(h))\n        return {"prob": p, "thresh": t, "binary": torch.sigmoid(self.k * (p - t)), "logit_prob": zp}\n\n    # ------------------------------------------------------------------ inferencia\n    @torch.no_grad()\n    def predict_proba(self, x):\n        """P (B, 1, H, W), sin importar el modo del modulo."""\n        self._validar(x)\n        return torch.sigmoid(self.cabeza_prob(self._caracteristicas(x)))\n\n    @torch.no_grad()\n    def predict_mask(self, x, variante="dbnet_at", valida=None):\n        """Mascara binaria (B, H, W) bool: P pasa por el posproceso de la variante (src.geometria_at.VARIANTES).\n        \'dbnet_at\' = C1 + C2 (expansion consistente en inglete); \'dbnet\' = el posproceso de DBNet (r\' = 1,5).\n        valida: bool (B, H, W) con la zona valida (sin relleno del letterbox); por defecto, toda la imagen."""\n        if variante not in G.VARIANTES:\n            raise ValueError(f"variante desconocida: {variante}; opciones: {sorted(G.VARIANTES)}")\n        p = self.predict_proba(x)[:, 0].cpu().numpy()\n        v = np.ones(p.shape, bool) if valida is None else np.asarray(valida, bool).reshape(p.shape)\n        post = G.post_de(variante)\n        return np.stack([post(p[i], v[i]) for i in range(len(p))])\n\n    def perdida(self, salida, objetivos, **pesos):\n        return perdida_db(salida, objetivos, **pesos)\n\n\n# ---------------------------------------------------------------------- perdida (copia de la de E2)\ndef bce_ohem(logits, gt, mask, razon=3):\n    """BCE con minado de negativos dificiles 1:razon (E2, modelos.bce_ohem; codigo oficial de DBNet)."""\n    perd = F.binary_cross_entropy_with_logits(logits, gt, reduction="none")\n    pos = (gt > 0.5) & (mask > 0.5)\n    neg = (gt <= 0.5) & (mask > 0.5)\n    n_pos = int(pos.sum())\n    # sin positivos (imagen sin rotulo) se toman 100 negativos, como en las implementaciones oficiales\n    n_neg = min(int(neg.sum()), razon * n_pos if n_pos else 100)\n    perd_neg = torch.topk(perd[neg], n_neg).values if n_neg else perd.new_zeros(0)\n    return (perd[pos].sum() + perd_neg.sum()) / (n_pos + n_neg + 1e-6)\n\n\ndef dice_perdida(p, t, m):\n    """1 - Dice enmascarado, con p, t y m del mismo tamano (E2, modelos.dice_perdida)."""\n    p, t = p * m, t * m\n    return 1 - (2 * (p * t).sum() + 1e-6) / ((p * p).sum() + (t * t).sum() + 1e-6)\n\n\ndef perdida_db(salida, objetivos, peso_s=5.0, peso_t=10.0, peso_b=1.0):\n    """L = 5 Ls + 10 Lt + Lb, la de DBNet segun el codigo oficial y E2.\n\n    salida: dict de UNetAT en modo entrenamiento. objetivos: dict de tensores (B, 1, H, W) con las claves de\n    geometria_at.at_objetivos (db_gt, db_mask, db_umbral, db_umbral_mask)."""\n    ls = bce_ohem(salida["logit_prob"], objetivos["db_gt"], objetivos["db_mask"])\n    lb = dice_perdida(salida["binary"], objetivos["db_gt"], objetivos["db_mask"])\n    m = objetivos["db_umbral_mask"]\n    lt = (torch.abs(salida["thresh"] - objetivos["db_umbral"]) * m).sum() / (m.sum() + 1e-6)\n    return peso_s * ls + peso_t * lt + peso_b * lb\n', 'training/entrenar_original.py': '"""Entrena UNetAT (el modelo original de E3) con el arnes del benchmark E2.\n\nReutiliza, importandolos por ruta desde el repositorio de E2 (solo lectura), el cargador de datos y los\nutilidades del arnes: dataset_ready (partition, load_example, metrics) y experimento_e2_colab (setup_seed,\ntraining_batches, make_optimizer, rng_state, restore_rng, atomic_json, atomic_torch, sha, digest, environment,\nverify_sources). Lo que NO se puede reutilizar tal cual y se reescribe aqui, igual en lo esencial:\n- train_model: construye el modelo desde MODELS_E2 y calcula la perdida con compute_loss, que no conocen a UNetAT.\n  Aqui es el mismo bucle (misma semilla y orden de lotes por epoca, mismo optimizador, mismas epocas y paciencia,\n  mismos checkpoints) con perdida_db sobre las etiquetas de DBNet o de DBNet-AT.\n- evaluate_validation: calculaba el Dice con P >= 0,5. Aqui la mascara sale del posproceso de la variante\n  (para dbnet_at, C2) y se promedia por grupo como alla; mismo fork_rng con la semilla 20261118.\n- verify_data y el preflight de E2: leen las imagenes desde repo_dl_imo/e2/VOC2012, que no existe en el\n  repositorio de solo lectura; aqui se verifican los mismos SHA-256 de imagenes y mascaras contra los CSV, leyendo\n  las imagenes desde --voc. El preflight de E2 (preflight.json) solo cubre sus 10 modelos; aqui hay uno propio:\n  un forward y backward con perdida finita antes de entrenar, anotado en preflight_original.json.\n\nReceta de la U-Net compacta de E2 (config/experimento_e2.json, modelos.unet): Adam, lr 1e-3, weight_decay 0, lote 2,\n30 epocas, paciencia 5, entrada 256, sin preentrenamiento, sin aumentos (load_example no los hace), seleccion por el\nDice medio por grupo en validacion; aqui, despues del posproceso.\n\nUso en GPU (Colab):\n    python entrenar_original.py --variante dbnet dbnet_at --seeds 11 23 37 53 71 --output /content/drive/MyDrive/e3_original\nHumo en CPU:\n    python entrenar_original.py --cpu-smoke --allow-local-output --max-images 8 --epochs 2 --seeds 11 \\\n        --variante dbnet_at --output C:\\\\Proyectos\\\\estudio\\\\datos\\\\deep_learning\\\\e3_humo\nLas otras 6 variantes de la ablacion 2^3 no se entrenan: salen por inferencia con los mismos checkpoints\n(geometria_at.VARIANTES).\n"""\nimport argparse\nimport copy\nimport hashlib\nimport json\nimport sys\nimport time\nfrom collections import defaultdict\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\n\nE3 = Path(__file__).resolve().parents[1]\nE2_SCRIPTS = Path(r"C:\\Proyectos\\estudio\\datos\\deep_learning\\repo_dl_imo\\e2\\tesis_imo_para_codex\\scripts")\nVOC = Path(r"C:\\Proyectos\\estudio\\datos\\hazmat13_drive")\nSEMILLA_VALIDACION = 20261118  # la de evaluate_validation de E2\nCLAVES_BINARIAS = ("db_gt", "db_mask", "db_umbral_mask")  # 0/1: se guardan en uint8\nH = DS = G = UNetAT = perdida_db = None  # se importan en cargar_arnes()\n\n\ndef cargar_arnes(e2_scripts):\n    """Importa el arnes de E2 por ruta y los modulos de E3. Falla con un mensaje claro si falta algo."""\n    global H, DS, G, UNetAT, perdida_db\n    e2_scripts = Path(e2_scripts).resolve()\n    if not (e2_scripts / "experimento_e2_colab.py").exists():\n        raise FileNotFoundError(f"No esta el arnes de E2 en {e2_scripts}")\n    for ruta in (str(e2_scripts), str(E3)):\n        if ruta not in sys.path:\n            sys.path.insert(0, ruta)\n    import experimento_e2_colab as h\n    import dataset_ready as ds\n    from src import geometria_at as g\n    from src.original_model import UNetAT as red, perdida_db as perdida\n    H, DS, G, UNetAT, perdida_db = h, ds, g, red, perdida\n    return e2_scripts\n\n\n# ------------------------------------------------------------------------------------------ datos y trazabilidad\ndef firma_codigo(e2_scripts):\n    """SHA-256 conjunto del codigo de E3 (src y este script) y de los archivos del arnes de E2 que se usan."""\n    e3 = {str(p.relative_to(E3)): H.sha(p) for p in sorted((E3 / "src").glob("*.py"))}\n    e3["training/entrenar_original.py"] = H.sha(Path(__file__))\n    e2 = {n: H.sha(Path(e2_scripts) / n) for n in\n          ("experimento_e2_colab.py", "dataset_ready.py", "modelos_base.py", "preparar_base.py", "inicio_voc.py")}\n    return {"e3": H.digest(e3), "e2_arnes": H.digest(e2), "opencv": G.cv2.__version__}\n\n\ndef verificar_datos(filas, voc):\n    """Lo que hace verify_data de E2, con las imagenes leidas desde voc: sin fuga de grupos entre particiones\n    (del test solo se leen los metadatos del CSV, como E2), SHA-256 de imagen y mascara de cada fila usada y\n    normalizacion a 256. Devuelve el resumen de hashes de los CSV y de las filas usadas."""\n    base = H.PROJECT / "resultados/preentrenamiento"\n    firma = {f: H.sha(base / f) for f in ["train.csv", "val.csv", "test.csv", "normalization_train.json"]}\n    grupos = {s: {r["group_id"] for r in DS.partition(s, allow_test=(s == "test"))} for s in ("train", "val", "test")}\n    for a, b in (("train", "val"), ("train", "test"), ("val", "test")):\n        if grupos[a] & grupos[b]:\n            raise RuntimeError(f"Fuga de grupos entre {a}/{b}")\n    usados = {}\n    for particion, rows in filas.items():\n        if not rows:\n            raise RuntimeError(f"Particion vacia: {particion}")\n        for row in rows:\n            for ruta, esperado in ((Path(voc) / row["image_path"], row["image_sha256"]),\n                                   (H.PROJECT / row["binary_mask_path"], row["binary_mask_sha256"])):\n                if H.sha(ruta) != esperado:\n                    raise RuntimeError(f"Datos modificados: {ruta}")\n            usados[row["id"]] = row["image_sha256"]\n    norma = json.loads((base / "normalization_train.json").read_text())\n    if norma["size"] != 256:\n        raise RuntimeError("Normalizacion no corresponde a 256")\n    return H.digest({"csv": firma, "filas_usadas": usados})\n\n\nclass Datos:\n    """Imagenes de una particion cargadas en RAM (load_example no hace aumentos: es determinista) y, para train,\n    los objetivos de cada variante, calculados una vez por imagen."""\n\n    def __init__(self, filas, voc, objetivos=None, nombre=""):\n        self.filas, self.x, self.y, self.v, self.obj = filas, {}, {}, {}, {}\n        t0 = time.perf_counter()\n        for fila in filas:\n            x, y, v = DS.load_example(fila, data_root=voc)\n            self.x[fila["id"]], self.y[fila["id"]], self.v[fila["id"]] = x, y, v\n            if objetivos is not None:\n                o = objetivos(y, v)\n                self.obj[fila["id"]] = {k: (a.astype(np.uint8) if k in CLAVES_BINARIAS else a) for k, a in o.items()}\n        print(f"[datos] {nombre}: {len(filas)} imagenes en RAM ({time.perf_counter() - t0:.1f} s)", flush=True)\n\n    def lote(self, filas, device):\n        """(x, objetivos) como tensores. x: (B, 3, H, W); cada objetivo: (B, 1, H, W) float32."""\n        ids = [f["id"] for f in filas]\n        x = torch.from_numpy(np.stack([self.x[i] for i in ids])).to(device)\n        obj = {k: torch.from_numpy(np.stack([self.obj[i][k] for i in ids]).astype(np.float32)).to(device)\n               for k in self.obj[ids[0]]}\n        return x, obj\n\n\n# ------------------------------------------------------------------------------------------ validacion\n@torch.no_grad()\ndef evaluar_validacion(modelo, datos, variante, device):\n    """Como evaluate_validation de E2 (Dice por imagen, promedio por grupo, luego promedio de los grupos), pero la\n    mascara sale de predict_mask (posproceso de la variante) en lugar de P >= 0,5."""\n    modelo.eval()\n    agrupado, detalle = defaultdict(list), []\n    dispositivos = [torch.cuda.current_device()] if device.type == "cuda" else []\n    with torch.random.fork_rng(devices=dispositivos):\n        torch.manual_seed(SEMILLA_VALIDACION)\n        for fila in datos.filas:\n            i = fila["id"]\n            x = torch.from_numpy(datos.x[i][None]).to(device)\n            mascara = modelo.predict_mask(x, variante, valida=datos.v[i][None])[0]\n            puntajes = DS.metrics(mascara.astype(np.float32), datos.y[i], datos.v[i])\n            agrupado[fila["group_id"]].append(puntajes["Dice"])\n            detalle.append({"id": i, "group_id": fila["group_id"], **puntajes})\n    return float(np.mean([np.mean(v) for v in agrupado.values()])), detalle\n\n\n# ------------------------------------------------------------------------------------------ entrenamiento\ndef preflight(variante, config, identity, train, device, destino):\n    """Un forward y backward con perdida y gradientes finitos antes de entrenar (sustituye al preflight de E2)."""\n    H.setup_seed(0)\n    modelo = UNetAT().to(device).train()\n    x, obj = train.lote(train.filas[:2], device)\n    perdida = perdida_db(modelo(x), obj)\n    perdida.backward()\n    ok = bool(torch.isfinite(perdida)) and all(p.grad is not None and bool(torch.isfinite(p.grad).all())\n                                                for p in modelo.parameters())\n    informe = {"variante": variante, "status": "passed" if ok else "failed", "perdida": float(perdida.detach()),\n               "identity": identity, "config_hash": H.digest(config), "environment": H.environment()}\n    H.atomic_json(Path(destino) / f"preflight_original_{variante}.json", informe)\n    if not ok:\n        raise RuntimeError(f"Preflight fallido ({variante}): perdida {float(perdida.detach())}")\n\n\ndef entrenar(variante, semilla, output, config, identity, device, train, val, parada=None):\n    """train_model de E2 con UNetAT. Mismo formato de config.json, last.pt, best_epoch_XXX.pt, history.json y\n    result.json (fingerprint, best_file, last_sha256, best_sha256)."""\n    nombre = f"unetat_{variante}"\n    H.setup_seed(semilla)\n    dest = Path(output) / f"{nombre}_seed{semilla}"\n    dest.mkdir(parents=True, exist_ok=True)\n    spec = {"name": nombre, "seed": semilla, "config": config, "identity": identity, "environment": H.environment(),\n            "modelo": {"k": 50.0, "cabeza": "1x1", "g0": G.G0 if variante == "dbnet_at" else 0.0,\n                       "perdida": "5*Ls + 10*Lt + Lb"}}\n    huella = H.digest(spec)\n    cfg_file = dest / "config.json"\n    if cfg_file.exists():\n        if json.loads(cfg_file.read_text())["fingerprint"] != huella:\n            raise RuntimeError(f"Corrida incompatible: {dest}. Usa una carpeta de experimento nueva.")\n    else:\n        H.atomic_json(cfg_file, {"fingerprint": huella, **spec})\n    last = dest / "last.pt"\n    if (dest / "result.json").exists():\n        res = json.loads((dest / "result.json").read_text())\n        if res.get("fingerprint") != huella or not res.get("completed"):\n            raise RuntimeError("Resultado incompatible")\n        if not last.exists() or not (dest / res["best_file"]).exists():\n            raise RuntimeError("Faltan checkpoints de corrida completada")\n        if H.sha(last) != res["last_sha256"] or H.sha(dest / res["best_file"]) != res["best_sha256"]:\n            raise RuntimeError("Checkpoint completado corrupto/modificado")\n        return res\n    modelo = UNetAT().to(device)  # tras setup_seed: mismos pesos iniciales de codificador/decodificador que la U-Net de E2\n    optimizador = H.make_optimizer(modelo, config["models"]["unet"])\n    inicio, mejor, sin_mejora, historia, mejor_archivo = 0, -1.0, 0, [], None\n    if last.exists():\n        guardado = torch.load(last, map_location="cpu", weights_only=False)\n        if guardado["fingerprint"] != huella:\n            raise RuntimeError("Checkpoint incompatible")\n        modelo.load_state_dict(guardado["model"])\n        optimizador.load_state_dict(guardado["optimizer"])\n        inicio, mejor, sin_mejora, historia, mejor_archivo = (guardado["epoch"] + 1, guardado["best"], guardado["stale"],\n                                                              guardado["history"], guardado["best_file"])\n        if not (dest / mejor_archivo).exists():\n            raise RuntimeError("Falta mejor checkpoint referenciado")\n        H.restore_rng(guardado["rng"])\n    if parada is not None and inicio >= parada and inicio < config["epochs"] and sin_mejora < config["patience"]:\n        return {"completed": False, "paused": True, "epochs_completed": inicio}\n    for epoca in range(inicio, config["epochs"]):\n        if sin_mejora >= config["patience"]:\n            break\n        modelo.train()\n        t0, suma, vistas = time.perf_counter(), 0.0, 0\n        for filas in H.training_batches(train.filas, semilla, epoca, config["batch_size"]):\n            x, obj = train.lote(filas, device)\n            optimizador.zero_grad(set_to_none=True)\n            perdida = perdida_db(modelo(x), obj)\n            if not torch.isfinite(perdida):\n                raise RuntimeError("Perdida no finita")\n            perdida.backward()\n            optimizador.step()\n            suma += perdida.item() * len(filas)\n            vistas += len(filas)\n        t_entreno = time.perf_counter() - t0\n        puntaje, detalle = evaluar_validacion(modelo, val, variante, device)\n        if not np.isfinite(puntaje):\n            raise RuntimeError("Validacion no finita")\n        if puntaje > mejor:\n            mejor, sin_mejora = puntaje, 0\n            mejor_archivo = f"best_epoch_{epoca + 1:03d}.pt"\n            H.atomic_torch(dest / mejor_archivo, {"model": modelo.state_dict(), "epoch": epoca, "fingerprint": huella,\n                                                  "validation": detalle})\n        else:\n            sin_mejora += 1\n        historia.append({"epoch": epoca + 1, "train_loss": suma / vistas, "val_group_Dice": puntaje,\n                         "seconds": time.perf_counter() - t0, "seconds_train": t_entreno})\n        # last apunta a un best inmutable ya escrito: un corte no cambia su significado\n        H.atomic_torch(last, {"fingerprint": huella, "epoch": epoca, "model": modelo.state_dict(),\n                              "optimizer": optimizador.state_dict(), "best": mejor, "stale": sin_mejora,\n                              "history": historia, "best_file": mejor_archivo, "rng": H.rng_state()})\n        H.atomic_json(dest / "history.json", historia)\n        for viejo in dest.glob("best_epoch_*.pt"):\n            if viejo.name != mejor_archivo:\n                viejo.unlink()\n        print(f"{nombre} seed={semilla} epoch={epoca + 1} loss={suma / vistas:.4f} val={puntaje:.5f} "\n              f"({historia[-1][\'seconds\']:.1f} s; entrenamiento {t_entreno:.1f} s)", flush=True)\n        if (parada is not None and epoca + 1 >= parada\n                and epoca + 1 < config["epochs"] and sin_mejora < config["patience"]):\n            return {"completed": False, "paused": True, "epochs_completed": epoca + 1}\n    res = {"completed": True, "fingerprint": huella, "model": nombre, "variante": variante, "seed": semilla,\n           "best_val_group_Dice": mejor, "epochs_completed": len(historia), "best_file": mejor_archivo,\n           "parameters": sum(p.numel() for p in modelo.parameters()),\n           "last_sha256": H.sha(last), "best_sha256": H.sha(dest / mejor_archivo)}\n    H.atomic_json(dest / "result.json", res)\n    return res\n\n\ndef main(argv=None):\n    ap = argparse.ArgumentParser(description=__doc__.split("\\n")[0])\n    ap.add_argument("--variante", nargs="+", default=["dbnet", "dbnet_at"], choices=["dbnet", "dbnet_at"],\n                    help="brazos que se entrenan; las otras 6 variantes salen por inferencia")\n    ap.add_argument("--seeds", nargs="+", type=int, help="por defecto, las del config de E2")\n    ap.add_argument("--output", required=True)\n    ap.add_argument("--config", default=None, help="por defecto, config/experimento_e2.json del arnes de E2")\n    ap.add_argument("--allow-local-output", action="store_true")\n    ap.add_argument("--cpu-smoke", action="store_true", help="humo en CPU; exige --max-images")\n    ap.add_argument("--max-images", type=int, help="usa solo las primeras N imagenes de train y de val (humo)")\n    ap.add_argument("--epochs", type=int, help="sobrescribe las epocas del config (humo)")\n    ap.add_argument("--pause-after-epoch", type=int,\n                    help="pausa después de esta época absoluta sin cambiar la receta; omitir para reanudar")\n    ap.add_argument("--voc", default=str(VOC), help="carpeta que contiene VOC2012/ (las imagenes)")\n    ap.add_argument("--e2-scripts", default=str(E2_SCRIPTS))\n    args = ap.parse_args(argv)\n    if args.pause_after_epoch is not None and args.pause_after_epoch < 1:\n        ap.error("--pause-after-epoch debe ser positivo")\n\n    e2_scripts = cargar_arnes(args.e2_scripts)\n    output = Path(args.output).resolve()\n    if not args.allow_local_output:\n        import subprocess\n        subprocess.run(["mountpoint", "-q", "/content/drive"], check=True)\n        if not output.is_relative_to(Path("/content/drive/MyDrive")):\n            raise RuntimeError("Salida debe estar en Drive")\n    if args.cpu_smoke:\n        if not args.max_images or args.max_images > 64:\n            raise RuntimeError("--cpu-smoke exige --max-images (hasta 64): una corrida completa en CPU no es un humo")\n        device = torch.device("cpu")\n    else:\n        if not torch.cuda.is_available():\n            raise RuntimeError("Este entrenamiento requiere GPU; para un humo en CPU usa --cpu-smoke --max-images N")\n        device = torch.device("cuda")\n        if args.max_images or args.epochs:\n            raise RuntimeError("--max-images y --epochs son solo para --cpu-smoke")\n    config = json.loads(Path(args.config or H.PROJECT / "config/experimento_e2.json").read_text())\n    if (config["pretrained"] or config["size"] != 256 or config["batch_size"] < 2 or config["threshold"] != .5\n            or config["epochs"] < 1 or config["patience"] < 1\n            or config["loss"] != "0.5_BCEWithLogits_plus_0.5_softDice_per_image_valid"):\n        raise ValueError("Configuracion no soportada")\n    config = copy.deepcopy(config)\n    if args.epochs:\n        config["epochs"] = args.epochs\n    if args.max_images:\n        config["max_images"] = args.max_images  # entra en la huella: un humo nunca se confunde con una corrida completa\n\n    H.verify_sources()\n    train_filas, val_filas = DS.partition("train"), DS.partition("val")\n    if args.max_images:\n        train_filas, val_filas = train_filas[:args.max_images], val_filas[:args.max_images]\n    identity = {**firma_codigo(e2_scripts),\n                "data": verificar_datos({"train": train_filas, "val": val_filas}, args.voc)}\n    output.mkdir(parents=True, exist_ok=True)\n    val = Datos(val_filas, args.voc, nombre="val")\n    resultados = []\n    for variante in args.variante:\n        train = Datos(train_filas, args.voc, G.objetivos_de(variante), nombre=f"train ({variante}, objetivos C1)"\n                      if variante == "dbnet_at" else f"train ({variante}, objetivos DBNet)")\n        preflight(variante, config, identity, train, device, output)\n        for semilla in args.seeds or config["seeds"]:\n            resultados.append(entrenar(variante, semilla, output, config, identity, device, train, val,\n                                       parada=args.pause_after_epoch))\n    print(json.dumps(resultados, indent=1))\n    return resultados\n\n\nif __name__ == "__main__":\n    main()\n', 'training/entrenar_compactas_e3.py': '"""Amplía a0498 con las cinco semillas E3; reutiliza exactamente el entrenador E2."""\nimport argparse\nimport copy\nimport hashlib\nimport json\nimport sys\nfrom pathlib import Path\nfrom types import SimpleNamespace\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\'--e2\', required=True)\n    ap.add_argument(\'--output\', required=True)\n    ap.add_argument(\'--smoke-voc\', help=\'Solo humo CPU: dos imágenes, una época, salida separada\')\n    ap.add_argument(\'--smoke-masks\', help=\'Raíz de máscaras existentes, solo para humo local\')\n    args = ap.parse_args()\n    sys.path.insert(0, str(Path(args.e2) / \'scripts\'))\n    import busqueda_arquitectura_e2 as B\n    import torch\n    e3 = Path(__file__).resolve().parents[1]\n    train, val = B.C.partition(\'train\'), B.C.partition(\'val\')\n    proto = json.loads((Path(args.e2) / \'config/experimento_e2.json\').read_text())\n    if args.smoke_voc:\n        train, val = [B.C.reubicar(rows[:2], args.smoke_voc, args.smoke_masks) for rows in [train, val]]\n        device = torch.device(\'cpu\')\n    else:\n        if not torch.cuda.is_available():\n            raise RuntimeError(\'Se requiere GPU para la campaña\')\n        device = torch.device(\'cuda\')\n    ident = B.identidad(SimpleNamespace(local=bool(args.smoke_voc)), train, val)\n    ident[\'extension_e3\'] = hashlib.sha256(Path(__file__).read_bytes()).hexdigest()\n    out = Path(args.output)\n    out.mkdir(parents=True, exist_ok=True)\n    for family in [\'fcn\', \'unet\', \'segnet\']:\n        template = json.loads((e3 / f\'config/plantilla_{family}_a0498.json\').read_text())[\'tarea\']\n        assert template[\'arq\'] == \'a0498\' and template[\'w\'] == [64, 64, 64]\n        assert template[\'b\'] == [0, 1, 0] and template[\'dropout_p\'] == .2\n        for seed in ([89] if args.smoke_voc else [89, 97, 101, 103, 107]):\n            task = copy.deepcopy(template)\n            task.update(semilla=seed, id=f\'{family}_a0498_s{seed}\')\n            if args.smoke_voc:\n                task.update(epocas=1, prueba_local=True)\n            B.entrenar(task, out, proto, ident, device, train, val)\n\n\nif __name__ == \'__main__\':\n    main()\n', 'tests/test_original_model.py': '"""Seis pruebas obligatorias del modelo original (UNetAT, e3/src/original_model.py). Ejecutar con:\n    pytest e3/tests          (o, sin pytest, python e3/tests/correr_sin_pytest.py)\n\n(i)   sobreajusta un lote diminuto sintetico (rombos y rectangulos sobre fondo, 64 x 64);\n(ii)  la misma semilla da las mismas salidas;\n(iii) formas de salida y entradas invalidas;\n(iv)  gradiente no nulo y finito en todos los parametros;\n(v)   la propiedad del teorema (diseno.md 3.5), verificada numericamente, y el inglete propio contra pyclipper;\n(vi)  guardar y cargar el state_dict da salidas iguales; exportacion ONNX con prueba de ida y vuelta.\n"""\nimport io\nimport json\nimport sys\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\nimport pytest\nimport torch\n\nE3 = Path(__file__).resolve().parents[1]\nif str(E3) not in sys.path:\n    sys.path.insert(0, str(E3))\nfrom src import geometria_at as G  # noqa: E402\nfrom src.original_model import UNetAT, perdida_db  # noqa: E402\n\nJSON_GEOMETRIA = E3.parent / "resultados" / "geometria_sintetica.json"\nTAM = 256\n\n\n# ---------------------------------------------------------------------------------------- utilidades\ndef _poligono(forma, puntos):\n    m = np.zeros(forma, np.uint8)\n    cv2.fillPoly(m, [(np.asarray(puntos, float) * 16).astype(np.int32)], 1, lineType=cv2.LINE_8, shift=4)\n    return m\n\n\ndef _rombo(lado, dx=0.0, forma=(TAM, TAM)):\n    h, c = lado / np.sqrt(2), forma[0] / 2 + dx\n    return _poligono(forma, [[c, c - h], [c + h, c], [c, c + h], [c - h, c]])\n\n\ndef _rect(alto, aspecto, dx=0.0, forma=(TAM, TAM)):\n    w, c = alto * aspecto, forma[0] / 2 + dx\n    return _poligono(forma, [[c - w / 2, c - alto / 2], [c + w / 2, c - alto / 2],\n                             [c + w / 2, c + alto / 2], [c - w / 2, c + alto / 2]])\n\n\ndef _dice(a, b):\n    a, b = a.astype(bool), b.astype(bool)\n    return 2.0 * (a & b).sum() / (a.sum() + b.sum())\n\n\ndef _oraculo(variante, y):\n    """Dice de la forma y al pasar el nucleo de su etiqueta por el posproceso de la variante (prediccion perfecta)."""\n    v = np.ones(y.shape, bool)\n    nucleo = G.objetivos_de(variante)(y, v)["db_gt"][0]\n    return _dice(G.post_de(variante)(nucleo, v), y)\n\n\ndef _lote_sintetico(n=4, tam=64):\n    """n imagenes tam x tam: un rombo o un rectangulo de color sobre fondo de otro color, con algo de ruido fijo.\n    Devuelve x (n, 3, tam, tam) float32, mascaras (n, tam, tam) uint8 y los objetivos de DBNet-AT como tensores."""\n    rng = np.random.RandomState(7)\n    xs, ys = [], []\n    for i in range(n):\n        y = np.zeros((tam, tam), np.uint8)\n        c = tam / 2 + rng.uniform(-6, 6, 2)\n        if i % 2 == 0:  # rombo de lado 18-24\n            h = rng.uniform(18, 24) / np.sqrt(2)\n            y = _poligono((tam, tam), [[c[0], c[1] - h], [c[0] + h, c[1]], [c[0], c[1] + h], [c[0] - h, c[1]]])\n        else:  # rectangulo de 24 a 32 por 12 a 16\n            w, a = rng.uniform(24, 32), rng.uniform(12, 16)\n            y = _poligono((tam, tam), [[c[0] - w / 2, c[1] - a / 2], [c[0] + w / 2, c[1] - a / 2],\n                                       [c[0] + w / 2, c[1] + a / 2], [c[0] - w / 2, c[1] + a / 2]])\n        fondo, frente = rng.uniform(-1, 0, 3), rng.uniform(0.3, 1, 3)\n        x = np.where(y[None] > 0, frente[:, None, None], fondo[:, None, None]) + 0.05 * rng.randn(3, tam, tam)\n        xs.append(x.astype(np.float32))\n        ys.append(y)\n    obj = [G.at_objetivos(y, np.ones(y.shape, bool)) for y in ys]\n    tensores = {k: torch.from_numpy(np.stack([o[k] for o in obj])) for k in obj[0]}\n    return torch.from_numpy(np.stack(xs)), np.stack(ys), tensores\n\n\n# ---------------------------------------------------------------------------------------- (i) sobreajuste\ndef test_i_sobreajusta_un_lote_diminuto():\n    x, ys, obj = _lote_sintetico()\n    modelo = UNetAT(semilla=0)\n    opt = torch.optim.Adam(modelo.parameters(), lr=3e-3)\n    modelo.train()\n    inicial = None\n    for paso in range(250):\n        opt.zero_grad()\n        perdida = perdida_db(modelo(x), obj)\n        perdida.backward()\n        opt.step()\n        inicial = float(perdida) if inicial is None else inicial\n    final = float(perdida)\n    assert final < 0.35 * inicial, f"la perdida no bajo lo suficiente: {inicial:.3f} -> {final:.3f}"\n    modelo.eval()\n    mascaras = modelo.predict_mask(x, "dbnet_at")\n    dice = np.mean([_dice(m, y) for m, y in zip(mascaras, ys)])\n    assert dice >= 0.85, f"Dice del lote sobreajustado: {dice:.3f}"\n\n\n# ---------------------------------------------------------------------------------------- (ii) determinismo\ndef test_ii_misma_semilla_mismas_salidas():\n    x = torch.from_numpy(np.random.RandomState(1).randn(2, 3, 32, 32).astype(np.float32))\n    a, b, c = UNetAT(semilla=5).eval(), UNetAT(semilla=5).eval(), UNetAT(semilla=6).eval()\n    assert torch.equal(a(x), b(x))\n    assert not torch.equal(a(x), c(x))\n    assert all(torch.equal(p, q) for p, q in zip(a.state_dict().values(), b.state_dict().values()))\n    # crear con semilla no toca el generador global\n    torch.manual_seed(123)\n    antes = torch.rand(1)\n    torch.manual_seed(123)\n    UNetAT(semilla=99)\n    assert torch.equal(antes, torch.rand(1))\n    # en entrenamiento tambien (no hay Dropout ni BatchNorm: el modelo es determinista)\n    sa, sb = UNetAT(semilla=5).train()(x), UNetAT(semilla=5).train()(x)\n    assert all(torch.equal(sa[k], sb[k]) for k in sa)\n\n\n# ---------------------------------------------------------------------------------------- (iii) formas y entradas\ndef test_iii_formas_y_entradas_invalidas():\n    modelo = UNetAT(semilla=0)\n    x = torch.zeros(2, 3, 64, 48)\n    modelo.train()\n    s = modelo(x)\n    assert set(s) >= {"prob", "thresh", "binary"}\n    for k in ("prob", "thresh", "binary", "logit_prob"):\n        assert s[k].shape == (2, 1, 64, 48), k\n    for k in ("prob", "thresh", "binary"):\n        assert float(s[k].min()) >= 0.0 and float(s[k].max()) <= 1.0\n    modelo.eval()\n    p = modelo(x)\n    assert isinstance(p, torch.Tensor) and p.shape == (2, 1, 64, 48)\n    assert torch.allclose(p, modelo.predict_proba(x))\n    assert modelo.predict_mask(x).shape == (2, 64, 48) and modelo.predict_mask(x).dtype == bool\n    # entradas invalidas: errores claros, no fallos oscuros dentro de una convolucion\n    with pytest.raises(ValueError, match="3 canales"):\n        modelo(torch.zeros(1, 1, 64, 64))\n    with pytest.raises(ValueError, match="3 canales"):\n        modelo(torch.zeros(1, 4, 64, 64))\n    with pytest.raises(ValueError, match="multiplos de 4"):\n        modelo(torch.zeros(1, 3, 66, 64))\n    with pytest.raises(ValueError, match="multiplos de 4"):\n        modelo.predict_proba(torch.zeros(1, 3, 64, 62))\n    with pytest.raises(ValueError, match="B, 3, H, W"):\n        modelo(torch.zeros(3, 64, 64))\n    with pytest.raises(ValueError, match="tensor"):\n        modelo(np.zeros((1, 3, 64, 64), np.float32))\n    with pytest.raises(ValueError, match="flotante"):\n        modelo(torch.zeros(1, 3, 64, 64, dtype=torch.int64))\n    with pytest.raises(ValueError, match="variante"):\n        modelo.predict_mask(x, "inexistente")\n    with pytest.raises(ValueError):\n        UNetAT(cabeza="5x5")\n\n\n# ---------------------------------------------------------------------------------------- (iv) gradientes\ndef test_iv_gradiente_no_nulo_y_finito_en_todos_los_parametros():\n    x, _, obj = _lote_sintetico(n=2)\n    for cabeza in ("1x1", "3x3"):\n        modelo = UNetAT(semilla=0, cabeza=cabeza).train()\n        perdida = perdida_db(modelo(x), obj)\n        assert torch.isfinite(perdida)\n        perdida.backward()\n        for nombre, p in modelo.named_parameters():\n            assert p.grad is not None, f"{cabeza}: {nombre} sin gradiente"\n            assert torch.isfinite(p.grad).all(), f"{cabeza}: {nombre} con gradiente no finito"\n            assert float(p.grad.abs().sum()) > 0, f"{cabeza}: {nombre} con gradiente nulo"\n    # derivada de la binarizacion: dB/dP = k B (1 - B), dB/dT = -k B (1 - B)\n    p = torch.tensor([0.4, 0.5, 0.6], requires_grad=True)\n    t = torch.tensor([0.5, 0.5, 0.5], requires_grad=True)\n    b = torch.sigmoid(50.0 * (p - t))\n    b.sum().backward()\n    esperado = 50.0 * b.detach() * (1 - b.detach())\n    assert torch.allclose(p.grad, esperado, atol=1e-5) and torch.allclose(t.grad, -esperado, atol=1e-5)\n\n\n# ---------------------------------------------------------------------------------------- (v) la propiedad\nSUB = [i / 8 for i in range(8)]  # las 8 posiciones subpixel de geometria_sintetica.py\n\n\ndef _medio(variante, formas):\n    return float(np.mean([_oraculo(variante, y) for y in formas]))\n\n\ndef test_v_expansion_consistente_reconstruye_rectangulos():\n    """Teorema de diseno.md 3.5 con la regla de DBNet (phi = 1, variante \'c2\'): la expansion consistente en inglete\n    reconstruye rectangulos de aspecto 1, 2 y 4 con Dice >= 0,99; la expansion fija de DBNet no (aspecto 4)."""\n    for aspecto in (1, 2, 4):\n        formas = [_rect(32, aspecto, d) for d in SUB]\n        assert _medio("c2", formas) >= 0.99, f"c2, aspecto {aspecto}"\n    dbnet4 = _medio("dbnet", [_rect(32, 4, d) for d in SUB])\n    assert dbnet4 < 0.90, f"DBNet en aspecto 4 deberia fallar: {dbnet4:.3f}"\n    # el cociente se degrada con el aspecto en DBNet y no en C2: es el corolario de 3.5\n    assert _medio("dbnet", [_rect(32, 1, d) for d in SUB]) > dbnet4\n\n\ndef test_v_expansion_consistente_reconstruye_rombos():\n    """Rombos de lado >= 10 px. El teorema es exacto en el continuo; en pixeles la inversa no es unica en +-1 px\n    (diseno.md 3.3 y 3.5), asi que el Dice >= 0,99 no se cumple en todos los tamanos con ninguna de las dos reglas.\n    Se verifica lo que si se sostiene: Dice >= 0,99 en los tamanos donde la discretizacion es exacta (10 y 64, las\n    dos reglas) y, en todos los lados de 10 a 96, un promedio >= 0,96 con la mejor de las dos y un minimo >= 0,88."""\n    lados = (10, 12, 16, 24, 32, 48, 64, 96)\n    dice = {v: {l: _medio(v, [_rombo(l, d) for d in SUB]) for l in lados} for v in ("c2", "dbnet_at")}\n    for variante in dice:\n        for lado in (10, 64):\n            assert dice[variante][lado] >= 0.99, f"{variante}, rombo {lado}: {dice[variante][lado]:.4f}"\n        assert min(dice[variante].values()) >= 0.88, (variante, dice[variante])\n    mejor = [max(dice["c2"][l], dice["dbnet_at"][l]) for l in lados]\n    assert np.mean(mejor) >= 0.96, mejor\n\n\ndef test_v_coincide_con_resultados_previos_con_pyclipper():\n    """Las cifras de entregable_3/resultados/geometria_sintetica.json se calcularon con el inglete de pyclipper; el\n    inglete propio las reproduce (tolerancia 0,005 sobre el promedio de las 8 posiciones subpixel)."""\n    if not JSON_GEOMETRIA.exists():\n        pytest.skip("falta resultados/geometria_sintetica.json")\n    previo = json.loads(JSON_GEOMETRIA.read_text())\n    casos = [("rectangulos_alto32_por_aspecto", "4", [_rect(32, 4, d) for d in SUB]),\n             ("rectangulos_alto32_por_aspecto", "2", [_rect(32, 2, d) for d in SUB]),\n             ("rombos_por_lado_px", "24", [_rombo(24, d) for d in SUB]),\n             ("rombos_por_lado_px", "10", [_rombo(10, d) for d in SUB])]\n    for grupo, clave, formas in casos:\n        for variante in ("dbnet", "c2", "dbnet_at"):\n            esperado = previo[grupo][clave][variante]["dice"]\n            obtenido = _medio(variante, formas)\n            assert abs(obtenido - esperado) <= 0.005, f"{grupo} {clave} {variante}: {obtenido:.4f} vs {esperado}"\n\n\ndef _inglete_pyclipper(c, d, limite=2.0, escala=16):\n    """Referencia: el desplazamiento en inglete de pyclipper (algoritmo de Vatti), sobre el mismo contorno."""\n    pyclipper = pytest.importorskip("pyclipper")\n    salida = np.zeros(c.shape, np.uint8)\n    off = pyclipper.PyclipperOffset(miter_limit=limite)\n    for k in G._contornos(c):\n        off.AddPath((k * escala).astype(np.int64).tolist(), pyclipper.JT_MITER, pyclipper.ET_CLOSEDPOLYGON)\n    for q in off.Execute(d * escala):\n        cv2.fillPoly(salida, [np.asarray(q, np.int32)], 1, lineType=cv2.LINE_8, shift=4)\n    return salida > 0\n\n\ndef _iou(a, b):\n    return (a & b).sum() / (a | b).sum()\n\n\ndef test_distancia_okr_es_medio_lado_corto_en_rectangulos():\n    """La Ec. 2 de OKR da S = min(a, b) en un nucleo rectangular a x b; la distancia por lado es S / 2 (el area se\n    cuenta en pixeles y el perimetro sobre el contorno de cv2, que pasa por los centros: de ahi la tolerancia)."""\n    for a, b in [(10, 40), (16, 16), (24, 60)]:\n        m = np.zeros((96, 96), bool)\n        m[20:20 + a, 10:10 + b] = True\n        assert abs(G.distancia_okr(m) - min(a, b) / 2) <= 1.0, (a, b, G.distancia_okr(m))\n    m = np.zeros((96, 96), bool)\n    m[30:50, 20:70] = True\n    v = np.ones_like(m)\n    salida = G.post_de("okr_sobre_at")(m.astype(np.float32), v)\n    assert salida.sum() > m.sum()  # la variante OKR corre y expande\n\n\ndef test_v_inglete_propio_coincide_con_pyclipper_en_convexos():\n    forma = (160, 160)\n    convexos = {\n        "cuadrado": [[50, 50], [100, 50], [100, 100], [50, 100]],\n        "rectangulo": [[30, 60], [130, 60], [130, 90], [30, 90]],\n        "rombo": [[80, 40], [120, 80], [80, 120], [40, 80]],\n        "hexagono": [[60, 40], [100, 40], [120, 80], [100, 120], [60, 120], [40, 80]],\n        "pentagono": [[80, 30], [125, 65], [110, 120], [50, 120], [35, 65]],\n        "triangulo_70": [[40, 120], [120, 120], [80, 50]],\n    }\n    for nombre, pts in convexos.items():\n        c = _poligono(forma, pts) > 0\n        for d in (2.0, 5.0, 9.0):\n            propio = G.rellenar_inglete(forma, G._contornos(c), d) > 0\n            ref = _inglete_pyclipper(c, d)\n            assert _iou(propio, ref) >= 0.98, f"{nombre}, d = {d}: IoU {_iou(propio, ref):.4f}"\n    # el limite de inglete: una punta de 30 grados se corta a 2 d de su vertice (aqui y en pyclipper)\n    agudo = _poligono(forma, [[20, 80], [140, 65], [140, 95]]) > 0\n    propio = G.rellenar_inglete(forma, G._contornos(agudo), 6.0) > 0\n    assert _iou(propio, _inglete_pyclipper(agudo, 6.0)) >= 0.95\n    # no convexo (forma de L): el camino de piezas coincide con pyclipper\n    ele = _poligono(forma, [[40, 40], [70, 40], [70, 90], [120, 90], [120, 120], [40, 120]]) > 0\n    propio = G.rellenar_inglete(forma, G._contornos(ele), 5.0) > 0\n    assert _iou(propio, _inglete_pyclipper(ele, 5.0)) >= 0.95\n    # d = 0 no cambia el poligono, y el desplazamiento nunca encoge\n    assert _iou(G.rellenar_inglete(forma, G._contornos(ele), 0.0) > 0, ele) >= 0.97\n\n\ndef test_v_variante_dbnet_equivale_a_e2():\n    """Con g0 = 0 y r\' fijo, la etiqueta y el posproceso son los de E2 (entregable_2/codigo/etiquetas.py)."""\n    e2 = E3.parents[1] / "entregable_2" / "codigo"\n    if not (e2 / "etiquetas.py").exists():\n        pytest.skip("falta entregable_2/codigo/etiquetas.py")\n    sys.path.insert(0, str(e2))\n    try:\n        import etiquetas as E2\n    finally:\n        sys.path.remove(str(e2))\n    y = np.maximum(_rombo(30, 0.3, (TAM, TAM)), _poligono((TAM, TAM), [[20, 20], [70, 20], [70, 40], [20, 40]]))\n    v = np.ones(y.shape, bool)\n    propio, e2o = G.db_objetivos(y, v), E2.db_objetivos(y, v)\n    for k in e2o:\n        assert np.allclose(propio[k], e2o[k]), k\n    nucleo = propio["db_gt"][0]\n    assert np.array_equal(G.db_post(nucleo, v), E2.db_post(nucleo, v))\n\n\n# ---------------------------------------------------------------------------------------- (vi) guardar y cargar\ndef test_vi_guardar_y_cargar_state_dict():\n    x = torch.from_numpy(np.random.RandomState(3).randn(2, 3, 32, 32).astype(np.float32))\n    a = UNetAT(semilla=1).eval()\n    buf = io.BytesIO()\n    torch.save(a.state_dict(), buf)\n    buf.seek(0)\n    b = UNetAT(semilla=2).eval()\n    assert not torch.equal(a(x), b(x))\n    b.load_state_dict(torch.load(buf, map_location="cpu", weights_only=True))\n    assert torch.equal(a(x), b(x))\n    assert torch.equal(a.predict_proba(x), b.predict_proba(x))\n\n\nclass _RedSinPosproceso(torch.nn.Module):\n    """La red tal cual (P, T, B), para exportar a ONNX: sin el posproceso, que es de numpy."""\n\n    def __init__(self, modelo):\n        super().__init__()\n        self.m = modelo\n\n    def forward(self, x):\n        h = self.m._caracteristicas(x)\n        p = torch.sigmoid(self.m.cabeza_prob(h))\n        t = torch.sigmoid(self.m.cabeza_umbral(h))\n        return p, t, torch.sigmoid(self.m.k * (p - t))\n\n\ndef test_vi_exportacion_onnx_ida_y_vuelta(tmp_path=None):\n    pytest.importorskip("onnx")  # torch.onnx.export lo exige aun con dynamo=False\n    ort = pytest.importorskip("onnxruntime")\n    import tempfile\n    modelo = UNetAT(semilla=4).eval()\n    red = _RedSinPosproceso(modelo).eval()\n    x = torch.from_numpy(np.random.RandomState(5).randn(1, 3, 64, 64).astype(np.float32))\n    with tempfile.TemporaryDirectory() as d:\n        ruta = str(Path(d) / "unetat.onnx")\n        torch.onnx.export(red, (x,), ruta, input_names=["imagen"], output_names=["prob", "thresh", "binary"],\n                          opset_version=17, dynamo=False)\n        sesion = ort.InferenceSession(ruta, providers=["CPUExecutionProvider"])\n        salidas = sesion.run(None, {"imagen": x.numpy()})\n    with torch.no_grad():\n        esperadas = red(x)\n    for onnx_, torch_ in zip(salidas, esperadas):\n        assert np.allclose(onnx_, torch_.numpy(), atol=1e-4)\n', 'tests/conftest.py': '"""Configuracion de pytest: pone e3/ en sys.path para importar `src` sin instalar el paquete."""\nimport sys\nfrom pathlib import Path\n\nE3 = Path(__file__).resolve().parents[1]\nif str(E3) not in sys.path:\n    sys.path.insert(0, str(E3))\n', 'experiments/auditar_campana.py': '"""Inventario por rutas explícitas, sin inferir carpetas ni cargar imágenes de test."""\nimport argparse\nimport csv\nimport hashlib\nimport json\nfrom pathlib import Path\n\nSEEDS = [11, 23, 37, 53, 71, 89, 97, 101, 103, 107]\nOTHERS = [\'transunet_2d\', \'pidnet_s\', \'vmunet\', \'segnext_t\', \'deeplabv3plus\', \'unetplusplus\']\n\n\ndef sha(p):\n    h = hashlib.sha256()\n    with p.open(\'rb\') as f:\n        for block in iter(lambda: f.read(1024 * 1024), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\'--campana\', type=Path, required=True)\n    ap.add_argument(\'--benchmark\', type=Path, required=True)\n    ap.add_argument(\'--compactas\', type=Path, required=True)\n    args = ap.parse_args()\n    targets = []\n    for s in SEEDS:\n        for arm in [\'dbnet\', \'dbnet_at\']:\n            targets.append((f\'unetat_{arm}\', s, args.campana / \'original\' / f\'unetat_{arm}_seed{s}\'))\n        for family in [\'fcn\', \'unet\', \'segnet\']:\n            root = args.compactas if s in SEEDS[:5] else args.campana / \'compactas\'\n            targets.append((family, s, root / \'fase2\' / f\'{family}_a0498_s{s}\'))\n        for model in OTHERS + [\'segformer_b0\']:\n            session = \'A\' if model in OTHERS[:2] else \'B\'\n            if model == \'segformer_b0\': session = \'C\'\n            root = args.benchmark if s in SEEDS[:5] and model != \'segformer_b0\' else args.campana / f\'rivales/sesion_{session}\'\n            targets.append((model, s, root / f\'{model}_seed{s}\'))\n    rows = []\n    for model, seed, directory in targets:\n        row = dict(modelo=model, semilla=seed, directorio=str(directory), estado=\'falta_resultado\', dice_val=\'\')\n        rp = directory / \'result.json\'\n        if rp.exists():\n            try:\n                r = json.loads(rp.read_text())\n                cfg = json.loads((directory / \'config.json\').read_text())\n                assert cfg[\'fingerprint\'] == r[\'fingerprint\']\n                assert r.get(\'completed\') is True and r.get(\'seed\', r.get(\'semilla\')) == seed\n                assert r.get(\'model\', r.get(\'familia\')) == model\n                if model in [\'fcn\', \'unet\', \'segnet\']:\n                    assert r[\'arq\'] == \'a0498\'\n                    assert cfg[\'tarea\'][\'w\'] == [64,64,64] and cfg[\'tarea\'][\'b\'] == [0,1,0]\n                if model.startswith(\'unetat_\'):\n                    assert cfg[\'identity\'][\'opencv\'] == \'4.11.0\'\n                if model == \'segformer_b0\':\n                    recipe = cfg[\'config\'][\'models\'][model]\n                    assert recipe[\'optimizer\']==\'AdamW\' and abs(recipe[\'lr\']-.0005573638936676344)<1e-14\n                    assert recipe[\'weight_decay\']==.0001\n                name = r[\'best_file\']\n                assert Path(name).name == name\n                assert sha(directory / name) == r[\'best_sha256\']\n                row.update(estado=\'verificado\', dice_val=r[\'best_val_group_Dice\'])\n            except (KeyError, ValueError, AssertionError, OSError) as e:\n                row[\'estado\'] = \'invalido:\' + type(e).__name__\n        rows.append(row)\n    args.campana.mkdir(parents=True, exist_ok=True)\n    with (args.campana / \'inventario_campana.csv\').open(\'w\', newline=\'\', encoding=\'utf-8\') as f:\n        writer = csv.DictWriter(f, fieldnames=list(rows[0]))\n        writer.writeheader(); writer.writerows(rows)\n    report = dict(esperadas=len(rows), verificadas=sum(r[\'estado\']==\'verificado\' for r in rows),\n                  test_evaluado=False, pendientes=[r for r in rows if r[\'estado\']!=\'verificado\'])\n    report[\'lista_para_revision_pretest\'] = report[\'verificadas\'] == report[\'esperadas\']\n    (args.campana / \'estado_campana.json\').write_text(json.dumps(report, indent=2), encoding=\'utf-8\')\n    print(json.dumps(report, indent=2))\n    return 0 if report[\'lista_para_revision_pretest\'] else 2\n\n\nif __name__ == \'__main__\':\n    raise SystemExit(main())\n', 'training/compactas_e2.py': '"""Arquitecturas compactas E2; copia fiel de definiciones, sin importar runners.\nProcedencia y hashes en config/fuentes_compactas.json. No es el mecanismo original E3.\n"""\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\ndef block(a, b):\n    return nn.Sequential(nn.Conv2d(a, b, 3, padding=1), nn.ReLU(),\n                         nn.Conv2d(b, b, 3, padding=1), nn.ReLU())\n\ndef _drop(activo, p):\n    return nn.Dropout(p) if activo else nn.Identity()\n\nclass FCNBusqueda(nn.Module):\n    """SmallFCN con anchos (w1, w2, w3) y dropout tras cada bloque. Mismo orden de creación de capas."""\n\n    def __init__(self, w, b, p=0.2):\n        super().__init__()\n        w1, w2, w3 = w\n        self.e1, self.e2, self.e3 = block(3, w1), block(w1, w2), block(w2, w3)\n        self.head = nn.Conv2d(w3, 1, 1)\n        self.r1, self.r2, self.r3 = (_drop(x, p) for x in b)\n\n    def forward(self, x):\n        a = self.r1(self.e1(x))\n        c = self.r2(self.e2(F.max_pool2d(a, 2)))\n        c = self.r3(self.e3(F.max_pool2d(c, 2)))\n        return F.interpolate(self.head(c), size=x.shape[-2:], mode=\'bilinear\', align_corners=False)\n\nclass UNetBusqueda(nn.Module):\n    """SmallUNet con anchos (w1, w2, w3); el decodificador espeja los anchos y el dropout."""\n\n    def __init__(self, w, b, p=0.2):\n        super().__init__()\n        w1, w2, w3 = w\n        self.e1, self.e2, self.e3 = block(3, w1), block(w1, w2), block(w2, w3)\n        self.d2, self.d1 = block(w3 + w2, w2), block(w2 + w1, w1)\n        self.head = nn.Conv2d(w1, 1, 1)\n        self.r1, self.r2, self.r3 = (_drop(x, p) for x in b)\n        self.s2, self.s1 = _drop(b[1], p), _drop(b[0], p)  # espejos de los niveles 2 y 1\n\n    def forward(self, x):\n        a = self.r1(self.e1(x))\n        m = self.r2(self.e2(F.max_pool2d(a, 2)))\n        c = self.r3(self.e3(F.max_pool2d(m, 2)))\n        d = self.s2(self.d2(torch.cat([F.interpolate(c, size=m.shape[-2:], mode=\'nearest\'), m], 1)))\n        d = self.s1(self.d1(torch.cat([F.interpolate(d, size=a.shape[-2:], mode=\'nearest\'), a], 1)))\n        return self.head(d)\n\nclass SegNetBusqueda(nn.Module):\n    """SmallSegNet con anchos (w1, w2, w3); reducción w3 -> w2 y decodificador espejo con unpooling."""\n\n    def __init__(self, w, b, p=0.2):\n        super().__init__()\n        w1, w2, w3 = w\n        self.e1, self.e2, self.e3 = block(3, w1), block(w1, w2), block(w2, w3)\n        self.reduce = nn.Conv2d(w3, w2, 1)\n        self.d2, self.d1 = block(w2, w1), block(w1, w1)\n        self.head = nn.Conv2d(w1, 1, 1)\n        self.r1, self.r2, self.r3 = (_drop(x, p) for x in b)\n        self.s2, self.s1 = _drop(b[1], p), _drop(b[0], p)\n\n    def forward(self, x):\n        a = self.r1(self.e1(x))\n        p, i1 = F.max_pool2d(a, 2, return_indices=True)\n        m = self.r2(self.e2(p))\n        p, i2 = F.max_pool2d(m, 2, return_indices=True)\n        c = self.reduce(self.r3(self.e3(p)))\n        d = self.s2(self.d2(F.max_unpool2d(c, i2, 2, output_size=m.shape)))\n        d = self.s1(self.d1(F.max_unpool2d(d, i1, 2, output_size=a.shape)))\n        return self.head(d)\n\nCLASES = {\'fcn\': FCNBusqueda, \'unet\': UNetBusqueda, \'segnet\': SegNetBusqueda}\n\ndef construir_red(familia, w, b, p):\n    return CLASES[familia](tuple(w), tuple(b), p)\n', 'experiments/evaluar_campana.py': '"""Predicciones por imagen en validación para la campaña íntegra, sin selección por test."""\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport sys\nfrom collections import defaultdict\nfrom pathlib import Path\n\n\ndef sha(p):\n    h = hashlib.sha256()\n    with p.open(\'rb\') as f:\n        for b in iter(lambda: f.read(1024*1024), b\'\'): h.update(b)\n    return h.hexdigest()\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\'--e2\', type=Path, required=True)\n    ap.add_argument(\'--e3\', type=Path, required=True)\n    ap.add_argument(\'--voc\', type=Path, required=True)\n    ap.add_argument(\'--campana\', type=Path, required=True)\n    ap.add_argument(\'--smoke\', action=\'store_true\', help=\'Solo prueba técnica, dos imágenes, salida separada\')\n    args = ap.parse_args()\n    sys.path[:0] = [str(args.e2/\'scripts\'), str(args.e3), str(args.e3/\'training\')]\n    import numpy as np\n    import torch\n    import cv2\n    from dataset_ready import load_example, partition, metrics\n    from modelos_e2 import MODELS_E2, main_logits\n    from compactas_e2 import construir_red\n    from src.original_model import UNetAT\n    assert cv2.__version__==\'4.11.0\'\n    state = json.loads((args.campana/\'estado_campana.json\').read_text())\n    if args.smoke:\n        assert state.get(\'smoke\') is True\n    else:\n        assert state[\'lista_para_revision_pretest\'], \'Completar y auditar primero las 120 corridas\'\n    with (args.campana/\'inventario_campana.csv\').open(encoding=\'utf-8\') as f:\n        runs=list(csv.DictReader(f))\n    assert (args.smoke or len(runs)==120) and all(r[\'estado\']==\'verificado\' for r in runs)\n    device=torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n    torch.set_num_threads(2)\n    torch.backends.cudnn.benchmark=False\n    torch.backends.cudnn.deterministic=True\n    rows=partition(\'val\')\n    if args.smoke: rows=rows[:2]\n    data=[(row,*load_example(row,data_root=args.voc)) for row in rows]\n    out=args.campana/(\'evaluacion_validacion_humo\' if args.smoke else \'evaluacion_validacion\'); out.mkdir(exist_ok=True)\n    summaries=[]\n    for run in runs:\n        folder=Path(run[\'directorio\']); name=run[\'modelo\']; seed=int(run[\'semilla\'])\n        result=json.loads((folder/\'result.json\').read_text())\n        cfg=json.loads((folder/\'config.json\').read_text())\n        checkpoint=folder/result[\'best_file\']; assert sha(checkpoint)==result[\'best_sha256\']\n        destination=out/f\'{name}_s{seed}.csv\'\n        stamp=destination.with_suffix(\'.json\')\n        identity={\'checkpoint_sha256\':result[\'best_sha256\'],\'script\':sha(Path(__file__)),\n                  \'data\':sha(args.e2/\'resultados/preentrenamiento/val.csv\'),\n                  \'normalizacion\':sha(args.e2/\'resultados/preentrenamiento/normalization_train.json\'),\n                  \'opencv\':cv2.__version__,\'torch\':torch.__version__, \'smoke\':args.smoke,\n                  \'dispositivo\':torch.cuda.get_device_name(0) if device.type==\'cuda\' else \'cpu\'}\n        if stamp.exists() and destination.exists():\n            previous=json.loads(stamp.read_text())\n            assert previous[\'identity\']==identity and previous[\'csv_sha256\']==sha(destination)\n            summaries.append(previous); continue\n        torch.manual_seed(20261118)\n        if name.startswith(\'unetat_\'):\n            model=UNetAT()\n        elif name in [\'fcn\',\'unet\',\'segnet\']:\n            t=cfg[\'tarea\']; model=construir_red(name,t[\'w\'],t[\'b\'],t[\'dropout_p\'])\n        else:\n            model=MODELS_E2[name]()\n        model.load_state_dict(torch.load(checkpoint,map_location=\'cpu\',weights_only=True)[\'model\'],strict=True)\n        model=model.to(device).eval(); predictions=[]; groups=defaultdict(list)\n        with torch.no_grad(), torch.random.fork_rng(devices=[torch.cuda.current_device()] if device.type==\'cuda\' else []):\n            torch.manual_seed(20261118)\n            for row,x,y,v in data:\n                tensor=torch.from_numpy(x[None]).to(device)\n                if name.startswith(\'unetat_\'):\n                    prob=model.predict_mask(tensor,name.removeprefix(\'unetat_\'),valida=v[None])[0].astype(np.float32)\n                else:\n                    prob=main_logits(model(tensor)).sigmoid()[0,0].cpu().numpy()\n                scores=metrics(prob,y,v)\n                predictions.append(dict(modelo=name,semilla=seed,id=row[\'id\'],group_id=row[\'group_id\'],**scores))\n                groups[row[\'group_id\']].append(scores[\'Dice\'])\n        with destination.open(\'w\',newline=\'\',encoding=\'utf-8\') as f:\n            writer=csv.DictWriter(f,fieldnames=list(predictions[0])); writer.writeheader(); writer.writerows(predictions)\n        value=float(np.mean([np.mean(g) for g in groups.values()]))\n        record={\'modelo\':name,\'semilla\':seed,\'identity\':identity,\'csv_sha256\':sha(destination),\n                \'imagenes\':len(predictions),\'grupos\':len(groups),\'dice_val\':value,\n                \'dice_guardado\':result[\'best_val_group_Dice\'],\n                \'diferencia_dice\':value-result[\'best_val_group_Dice\'],\'test_evaluado\':False}\n        stamp.write_text(json.dumps(record,indent=2),encoding=\'utf-8\'); summaries.append(record)\n        print(name,seed,\'Dice\',value,\'delta\',record[\'diferencia_dice\'],flush=True)\n        del model\n        if device.type==\'cuda\':torch.cuda.empty_cache()\n    (out/\'resumen.json\').write_text(json.dumps(summaries,indent=2),encoding=\'utf-8\')\n\n\nif __name__==\'__main__\':main()\n', 'config/hpo_original_t0.json': '{\n  "schema": 2,\n  "experiment": "e3_hpo_original_t0",\n  "pretrained": false,\n  "size": 256,\n  "batch_size": 2,\n  "epochs": 10,\n  "patience": 3,\n  "seeds": [\n    11\n  ],\n  "threshold": 0.5,\n  "loss": "0.5_BCEWithLogits_plus_0.5_softDice_per_image_valid",\n  "pidnet_aux_weight": 0.4,\n  "pidnet_boundary_weight": 1.0,\n  "models": {\n    "fcn": {\n      "optimizer": "Adam",\n      "lr": 0.001,\n      "weight_decay": 0.0\n    },\n    "segnet": {\n      "optimizer": "Adam",\n      "lr": 0.001,\n      "weight_decay": 0.0\n    },\n    "unet": {\n      "optimizer": "Adam",\n      "lr": 0.001,\n      "weight_decay": 0.0\n    },\n    "deeplabv3plus": {\n      "optimizer": "AdamW",\n      "lr": 0.0005,\n      "weight_decay": 0.01\n    },\n    "unetplusplus": {\n      "optimizer": "Adam",\n      "lr": 0.0005,\n      "weight_decay": 0.0\n    },\n    "pidnet_s": {\n      "optimizer": "AdamW",\n      "lr": 0.001,\n      "weight_decay": 0.01\n    },\n    "segnext_t": {\n      "optimizer": "AdamW",\n      "lr": 0.0001,\n      "weight_decay": 0.01\n    },\n    "segformer_b0": {\n      "optimizer": "AdamW",\n      "lr": 6e-05,\n      "weight_decay": 0.01\n    },\n    "transunet_2d": {\n      "optimizer": "SGD",\n      "lr": 0.001,\n      "momentum": 0.9,\n      "weight_decay": 0.0001\n    },\n    "vmunet": {\n      "optimizer": "AdamW",\n      "lr": 0.001,\n      "weight_decay": 0.01\n    }\n  },\n  "nota_e3": "Búsqueda exploratoria del original con el presupuesto de la del E2 (10 épocas, paciencia 3, semilla 11). models.unet es la receta del original."\n}', 'config/hpo_original_t1.json': '{\n  "schema": 2,\n  "experiment": "e3_hpo_original_t1",\n  "pretrained": false,\n  "size": 256,\n  "batch_size": 2,\n  "epochs": 10,\n  "patience": 3,\n  "seeds": [\n    11\n  ],\n  "threshold": 0.5,\n  "loss": "0.5_BCEWithLogits_plus_0.5_softDice_per_image_valid",\n  "pidnet_aux_weight": 0.4,\n  "pidnet_boundary_weight": 1.0,\n  "models": {\n    "fcn": {\n      "optimizer": "Adam",\n      "lr": 0.001,\n      "weight_decay": 0.0\n    },\n    "segnet": {\n      "optimizer": "Adam",\n      "lr": 0.001,\n      "weight_decay": 0.0\n    },\n    "unet": {\n      "optimizer": "Adam",\n      "lr": 0.00010193470686927321,\n      "weight_decay": 0.0001\n    },\n    "deeplabv3plus": {\n      "optimizer": "AdamW",\n      "lr": 0.0005,\n      "weight_decay": 0.01\n    },\n    "unetplusplus": {\n      "optimizer": "Adam",\n      "lr": 0.0005,\n      "weight_decay": 0.0\n    },\n    "pidnet_s": {\n      "optimizer": "AdamW",\n      "lr": 0.001,\n      "weight_decay": 0.01\n    },\n    "segnext_t": {\n      "optimizer": "AdamW",\n      "lr": 0.0001,\n      "weight_decay": 0.01\n    },\n    "segformer_b0": {\n      "optimizer": "AdamW",\n      "lr": 6e-05,\n      "weight_decay": 0.01\n    },\n    "transunet_2d": {\n      "optimizer": "SGD",\n      "lr": 0.001,\n      "momentum": 0.9,\n      "weight_decay": 0.0001\n    },\n    "vmunet": {\n      "optimizer": "AdamW",\n      "lr": 0.001,\n      "weight_decay": 0.01\n    }\n  },\n  "nota_e3": "Búsqueda exploratoria del original con el presupuesto de la del E2 (10 épocas, paciencia 3, semilla 11). models.unet es la receta del original."\n}', 'config/hpo_original_t2.json': '{\n  "schema": 2,\n  "experiment": "e3_hpo_original_t2",\n  "pretrained": false,\n  "size": 256,\n  "batch_size": 2,\n  "epochs": 10,\n  "patience": 3,\n  "seeds": [\n    11\n  ],\n  "threshold": 0.5,\n  "loss": "0.5_BCEWithLogits_plus_0.5_softDice_per_image_valid",\n  "pidnet_aux_weight": 0.4,\n  "pidnet_boundary_weight": 1.0,\n  "models": {\n    "fcn": {\n      "optimizer": "Adam",\n      "lr": 0.001,\n      "weight_decay": 0.0\n    },\n    "segnet": {\n      "optimizer": "Adam",\n      "lr": 0.001,\n      "weight_decay": 0.0\n    },\n    "unet": {\n      "optimizer": "Adam",\n      "lr": 0.009289398227793906,\n      "weight_decay": 0.0001\n    },\n    "deeplabv3plus": {\n      "optimizer": "AdamW",\n      "lr": 0.0005,\n      "weight_decay": 0.01\n    },\n    "unetplusplus": {\n      "optimizer": "Adam",\n      "lr": 0.0005,\n      "weight_decay": 0.0\n    },\n    "pidnet_s": {\n      "optimizer": "AdamW",\n      "lr": 0.001,\n      "weight_decay": 0.01\n    },\n    "segnext_t": {\n      "optimizer": "AdamW",\n      "lr": 0.0001,\n      "weight_decay": 0.01\n    },\n    "segformer_b0": {\n      "optimizer": "AdamW",\n      "lr": 6e-05,\n      "weight_decay": 0.01\n    },\n    "transunet_2d": {\n      "optimizer": "SGD",\n      "lr": 0.001,\n      "momentum": 0.9,\n      "weight_decay": 0.0001\n    },\n    "vmunet": {\n      "optimizer": "AdamW",\n      "lr": 0.001,\n      "weight_decay": 0.01\n    }\n  },\n  "nota_e3": "Búsqueda exploratoria del original con el presupuesto de la del E2 (10 épocas, paciencia 3, semilla 11). models.unet es la receta del original."\n}', 'config/hpo_original_t3.json': '{\n  "schema": 2,\n  "experiment": "e3_hpo_original_t3",\n  "pretrained": false,\n  "size": 256,\n  "batch_size": 2,\n  "epochs": 10,\n  "patience": 3,\n  "seeds": [\n    11\n  ],\n  "threshold": 0.5,\n  "loss": "0.5_BCEWithLogits_plus_0.5_softDice_per_image_valid",\n  "pidnet_aux_weight": 0.4,\n  "pidnet_boundary_weight": 1.0,\n  "models": {\n    "fcn": {\n      "optimizer": "Adam",\n      "lr": 0.001,\n      "weight_decay": 0.0\n    },\n    "segnet": {\n      "optimizer": "Adam",\n      "lr": 0.001,\n      "weight_decay": 0.0\n    },\n    "unet": {\n      "optimizer": "Adam",\n      "lr": 0.0011899757998003779,\n      "weight_decay": 0.0\n    },\n    "deeplabv3plus": {\n      "optimizer": "AdamW",\n      "lr": 0.0005,\n      "weight_decay": 0.01\n    },\n    "unetplusplus": {\n      "optimizer": "Adam",\n      "lr": 0.0005,\n      "weight_decay": 0.0\n    },\n    "pidnet_s": {\n      "optimizer": "AdamW",\n      "lr": 0.001,\n      "weight_decay": 0.01\n    },\n    "segnext_t": {\n      "optimizer": "AdamW",\n      "lr": 0.0001,\n      "weight_decay": 0.01\n    },\n    "segformer_b0": {\n      "optimizer": "AdamW",\n      "lr": 6e-05,\n      "weight_decay": 0.01\n    },\n    "transunet_2d": {\n      "optimizer": "SGD",\n      "lr": 0.001,\n      "momentum": 0.9,\n      "weight_decay": 0.0001\n    },\n    "vmunet": {\n      "optimizer": "AdamW",\n      "lr": 0.001,\n      "weight_decay": 0.01\n    }\n  },\n  "nota_e3": "Búsqueda exploratoria del original con el presupuesto de la del E2 (10 épocas, paciencia 3, semilla 11). models.unet es la receta del original."\n}', 'config/plantilla_fcn_a0498.json': '{\n  "fingerprint": "7338ec31c25eeff5d68c9773b7a946d04d15a24b28a5f936a27f41cb3fa4ae8d",\n  "tarea": {\n    "id": "fcn_a0498_s11",\n    "bloque": "fase2",\n    "familia": "fcn",\n    "arq": "a0498",\n    "w": [\n      64,\n      64,\n      64\n    ],\n    "b": [\n      0,\n      1,\n      0\n    ],\n    "dropout_p": 0.2,\n    "semilla": 11,\n    "optim": {\n      "optimizer": "Adam",\n      "lr": 0.001,\n      "weight_decay": 0.0\n    },\n    "epocas": 30,\n    "paciencia": 5,\n    "conservar": "best",\n    "macs": 4039376896,\n    "macs_efectivos": 4039376896,\n    "px": 256,\n    "fraccion_train": 1.0,\n    "semilla_subconjunto": 20261123\n  },\n  "protocolo": {\n    "batch_size": 2,\n    "size": 256,\n    "threshold": 0.5\n  },\n  "identidad": {\n    "source": "38fdd50dad06ab7993d1328100e61ee839336cc63d1cabfbf540736404e8402f",\n    "data": "a3c47c120862424d8f402154176723c1e6f9055cb4a50d2966d637c44b32f206"\n  },\n  "entorno": {\n    "packages": {\n      "torch": "2.5.1+cu121",\n      "torchvision": "0.20.1+cu121",\n      "numpy": "1.26.4",\n      "Pillow": "11.1.0",\n      "segmentation-models-pytorch": "0.5.0",\n      "timm": "1.0.15",\n      "scipy": "1.15.2",\n      "ml-collections": "1.0.0",\n      "einops": "0.8.1",\n      "mamba-ssm": null,\n      "causal-conv1d": null,\n      "triton": "3.1.0",\n      "transformers": null\n    },\n    "python": "3.11.11",\n    "cuda": "12.1",\n    "device": "NVIDIA L4"\n  },\n  "filas": {\n    "train": 1015,\n    "val": 218\n  }\n}', 'config/plantilla_unet_a0498.json': '{\n  "fingerprint": "31ecbe680c0d96e9305f99fd8175b71b52d1263895979c1dceb2366dd1f78143",\n  "tarea": {\n    "id": "unet_a0498_s11",\n    "bloque": "fase2",\n    "familia": "unet",\n    "arq": "a0498",\n    "w": [\n      64,\n      64,\n      64\n    ],\n    "b": [\n      0,\n      1,\n      0\n    ],\n    "dropout_p": 0.2,\n    "semilla": 11,\n    "optim": {\n      "optimizer": "Adam",\n      "lr": 0.0003,\n      "weight_decay": 0.0\n    },\n    "epocas": 30,\n    "paciencia": 5,\n    "conservar": "best",\n    "macs": 13103005696,\n    "macs_efectivos": 13103005696,\n    "px": 256,\n    "fraccion_train": 1.0,\n    "semilla_subconjunto": 20261123\n  },\n  "protocolo": {\n    "batch_size": 2,\n    "size": 256,\n    "threshold": 0.5\n  },\n  "identidad": {\n    "source": "38fdd50dad06ab7993d1328100e61ee839336cc63d1cabfbf540736404e8402f",\n    "data": "a3c47c120862424d8f402154176723c1e6f9055cb4a50d2966d637c44b32f206"\n  },\n  "entorno": {\n    "packages": {\n      "torch": "2.5.1+cu121",\n      "torchvision": "0.20.1+cu121",\n      "numpy": "1.26.4",\n      "Pillow": "11.1.0",\n      "segmentation-models-pytorch": "0.5.0",\n      "timm": "1.0.15",\n      "scipy": "1.15.2",\n      "ml-collections": "1.0.0",\n      "einops": "0.8.1",\n      "mamba-ssm": null,\n      "causal-conv1d": null,\n      "triton": "3.1.0",\n      "transformers": null\n    },\n    "python": "3.11.11",\n    "cuda": "12.1",\n    "device": "NVIDIA L4"\n  },\n  "filas": {\n    "train": 1015,\n    "val": 218\n  }\n}', 'config/plantilla_segnet_a0498.json': '{\n  "fingerprint": "272c1a64e40f9a8ab33737b42f73317d0cf322ec6c57580019c6ce27b03d7e5d",\n  "tarea": {\n    "id": "segnet_a0498_s11",\n    "bloque": "fase2",\n    "familia": "segnet",\n    "arq": "a0498",\n    "w": [\n      64,\n      64,\n      64\n    ],\n    "b": [\n      0,\n      1,\n      0\n    ],\n    "dropout_p": 0.2,\n    "semilla": 11,\n    "optim": {\n      "optimizer": "Adam",\n      "lr": 0.0003,\n      "weight_decay": 0.0\n    },\n    "epocas": 30,\n    "paciencia": 5,\n    "conservar": "best",\n    "macs": 10099884032,\n    "macs_efectivos": 10099884032,\n    "px": 256,\n    "fraccion_train": 1.0,\n    "semilla_subconjunto": 20261123\n  },\n  "protocolo": {\n    "batch_size": 2,\n    "size": 256,\n    "threshold": 0.5\n  },\n  "identidad": {\n    "source": "38fdd50dad06ab7993d1328100e61ee839336cc63d1cabfbf540736404e8402f",\n    "data": "a3c47c120862424d8f402154176723c1e6f9055cb4a50d2966d637c44b32f206"\n  },\n  "entorno": {\n    "packages": {\n      "torch": "2.5.1+cu121",\n      "torchvision": "0.20.1+cu121",\n      "numpy": "1.26.4",\n      "Pillow": "11.1.0",\n      "segmentation-models-pytorch": "0.5.0",\n      "timm": "1.0.15",\n      "scipy": "1.15.2",\n      "ml-collections": "1.0.0",\n      "einops": "0.8.1",\n      "mamba-ssm": null,\n      "causal-conv1d": null,\n      "triton": "3.1.0",\n      "transformers": null\n    },\n    "python": "3.11.11",\n    "cuda": "12.1",\n    "device": "NVIDIA L4"\n  },\n  "filas": {\n    "train": 1015,\n    "val": 218\n  }\n}'}
FIRMAS = {'src/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'src/geometria_at.py': '7dba12fbba9333b04c4ce1ae96f305cdcb7c0865337d722b8c77f3538fd4676c', 'src/original_model.py': '395340738602521a528e3c14cc662e0cb3f737954d1b055a9e82c0b605fdcbff', 'training/entrenar_original.py': '3e879677394023a14cdf7fae50906d74cf37d2563bd88ec597a610d9b20325e1', 'training/entrenar_compactas_e3.py': 'c1a443e875f8fbe53056b70d7cf16dc898aab71abcdb1d63a6f08944648a004e', 'tests/test_original_model.py': '4aebc1187faea88eb58678a328cfef2d935b271a38ab5ff1c88458d53adad6a3', 'tests/conftest.py': '63827836fe51b1624503a03ce53b039f1dc9cca54b74222112a9782bb8d70a1c', 'experiments/auditar_campana.py': '8e6579b45c86967ae2d5ba23a0afca0e22e6a56b5dedfd58f06055dfcaa6ed54', 'training/compactas_e2.py': '3e51665e87a619d57ffa243f484a8daee475ceed9e60c129cc93abcbe35c256b', 'experiments/evaluar_campana.py': '349967a88e1393b39730dfb120b66506789dc267a1a2231d82b75d9b276d473a', 'config/hpo_original_t0.json': '61c4b3295c2496959b4daa570b5a28ab59bf321af744ab327656df4d92da8ea6', 'config/hpo_original_t1.json': '39429a75881401b332d82ac1eb6076bb0095a1a99d772cd45b4e46e3fc2a9bfb', 'config/hpo_original_t2.json': '213007613219b84623f1363de7aed6865a5e66d64d45be7d875a78d2a3c92943', 'config/hpo_original_t3.json': '668cb072a40671320096049ef86b6c9f9b07b478c4897e2163db901e4d0bef06', 'config/plantilla_fcn_a0498.json': 'f5722ac599f1a8884f12c886c2814c19a09bb5baf07fa4e2982620c2fbdf9ab0', 'config/plantilla_unet_a0498.json': 'de8a04e36145782352ed2641f960f696ff751e718023eee2802131ab32504df7', 'config/plantilla_segnet_a0498.json': '6cf74cd4d1bfce00f465577b162b482bc0d3d35c0464d85d649dab8caea45c94'}
HISTORICO = '{\n "rombos_por_lado_px": {\n  "6": {\n   "area_px": 41,\n   "dbnet": {\n    "dice": 0.9526,\n    "nucleo_px": 15\n   },\n   "c1_rfijo": {\n    "dice": 0.5348,\n    "nucleo_px": 44\n   },\n   "c2_redondo": {\n    "dice": 0.8332,\n    "nucleo_px": 15\n   },\n   "c2": {\n    "dice": 1.0,\n    "nucleo_px": 15\n   },\n   "at_redondo": {\n    "dice": 0.931,\n    "nucleo_px": 44\n   },\n   "dbnet_at": {\n    "dice": 0.6421,\n    "nucleo_px": 44\n   }\n  },\n  "8": {\n   "area_px": 85,\n   "dbnet": {\n    "dice": 0.8683,\n    "nucleo_px": 35\n   },\n   "c1_rfijo": {\n    "dice": 0.6863,\n    "nucleo_px": 54\n   },\n   "c2_redondo": {\n    "dice": 0.8683,\n    "nucleo_px": 35\n   },\n   "c2": {\n    "dice": 0.947,\n    "nucleo_px": 35\n   },\n   "at_redondo": {\n    "dice": 0.9536,\n    "nucleo_px": 54\n   },\n   "dbnet_at": {\n    "dice": 0.947,\n    "nucleo_px": 54\n   }\n  },\n  "10": {\n   "area_px": 113,\n   "dbnet": {\n    "dice": 0.9823,\n    "nucleo_px": 42\n   },\n   "c1_rfijo": {\n    "dice": 0.7801,\n    "nucleo_px": 62\n   },\n   "c2_redondo": {\n    "dice": 0.9823,\n    "nucleo_px": 42\n   },\n   "c2": {\n    "dice": 1.0,\n    "nucleo_px": 42\n   },\n   "at_redondo": {\n    "dice": 0.9823,\n    "nucleo_px": 62\n   },\n   "dbnet_at": {\n    "dice": 1.0,\n    "nucleo_px": 62\n   }\n  },\n  "12": {\n   "area_px": 145,\n   "dbnet": {\n    "dice": 0.9027,\n    "nucleo_px": 70\n   },\n   "c1_rfijo": {\n    "dice": 0.7478,\n    "nucleo_px": 95\n   },\n   "c2_redondo": {\n    "dice": 0.9253,\n    "nucleo_px": 70\n   },\n   "c2": {\n    "dice": 0.9352,\n    "nucleo_px": 70\n   },\n   "at_redondo": {\n    "dice": 0.9196,\n    "nucleo_px": 95\n   },\n   "dbnet_at": {\n    "dice": 0.9352,\n    "nucleo_px": 95\n   }\n  },\n  "16": {\n   "area_px": 265,\n   "dbnet": {\n    "dice": 0.9352,\n    "nucleo_px": 103\n   },\n   "c1_rfijo": {\n    "dice": 0.8737,\n    "nucleo_px": 122\n   },\n   "c2_redondo": {\n    "dice": 0.911,\n    "nucleo_px": 103\n   },\n   "c2": {\n    "dice": 0.9689,\n    "nucleo_px": 103\n   },\n   "at_redondo": {\n    "dice": 0.9839,\n    "nucleo_px": 122\n   },\n   "dbnet_at": {\n    "dice": 1.0,\n    "nucleo_px": 122\n   }\n  },\n  "24": {\n   "area_px": 613,\n   "dbnet": {\n    "dice": 0.9509,\n    "nucleo_px": 218\n   },\n   "c1_rfijo": {\n    "dice": 0.9039,\n    "nucleo_px": 262\n   },\n   "c2_redondo": {\n    "dice": 0.9509,\n    "nucleo_px": 218\n   },\n   "c2": {\n    "dice": 1.0,\n    "nucleo_px": 218\n   },\n   "at_redondo": {\n    "dice": 0.9546,\n    "nucleo_px": 262\n   },\n   "dbnet_at": {\n    "dice": 0.9515,\n    "nucleo_px": 262\n   }\n  },\n  "32": {\n   "area_px": 1105,\n   "dbnet": {\n    "dice": 0.969,\n    "nucleo_px": 380\n   },\n   "c1_rfijo": {\n    "dice": 0.9483,\n    "nucleo_px": 402\n   },\n   "c2_redondo": {\n    "dice": 0.969,\n    "nucleo_px": 380\n   },\n   "c2": {\n    "dice": 0.9842,\n    "nucleo_px": 380\n   },\n   "at_redondo": {\n    "dice": 0.9747,\n    "nucleo_px": 402\n   },\n   "dbnet_at": {\n    "dice": 1.0,\n    "nucleo_px": 402\n   }\n  },\n  "48": {\n   "area_px": 2381,\n   "dbnet": {\n    "dice": 0.969,\n    "nucleo_px": 835\n   },\n   "c1_rfijo": {\n    "dice": 0.9687,\n    "nucleo_px": 835\n   },\n   "c2_redondo": {\n    "dice": 0.9689,\n    "nucleo_px": 835\n   },\n   "c2": {\n    "dice": 0.975,\n    "nucleo_px": 835\n   },\n   "at_redondo": {\n    "dice": 0.9683,\n    "nucleo_px": 835\n   },\n   "dbnet_at": {\n    "dice": 1.0,\n    "nucleo_px": 835\n   }\n  },\n  "64": {\n   "area_px": 4141,\n   "dbnet": {\n    "dice": 0.9716,\n    "nucleo_px": 1436\n   },\n   "c1_rfijo": {\n    "dice": 0.9714,\n    "nucleo_px": 1438\n   },\n   "c2_redondo": {\n    "dice": 0.9718,\n    "nucleo_px": 1436\n   },\n   "c2": {\n    "dice": 1.0,\n    "nucleo_px": 1436\n   },\n   "at_redondo": {\n    "dice": 0.9714,\n    "nucleo_px": 1438\n   },\n   "dbnet_at": {\n    "dice": 1.0,\n    "nucleo_px": 1438\n   }\n  },\n  "96": {\n   "area_px": 9385,\n   "dbnet": {\n    "dice": 0.9684,\n    "nucleo_px": 3270\n   },\n   "c1_rfijo": {\n    "dice": 0.9684,\n    "nucleo_px": 3270\n   },\n   "c2_redondo": {\n    "dice": 0.9607,\n    "nucleo_px": 3270\n   },\n   "c2": {\n    "dice": 0.9873,\n    "nucleo_px": 3270\n   },\n   "at_redondo": {\n    "dice": 0.9683,\n    "nucleo_px": 3270\n   },\n   "dbnet_at": {\n    "dice": 0.9873,\n    "nucleo_px": 3270\n   }\n  }\n },\n "rectangulos_alto32_por_aspecto": {\n  "1": {\n   "dbnet": {\n    "dice": 0.9775,\n    "nucleo_px": 361\n   },\n   "c1_rfijo": {\n    "dice": 0.9079,\n    "nucleo_px": 441\n   },\n   "c2_redondo": {\n    "dice": 0.9775,\n    "nucleo_px": 361\n   },\n   "c2": {\n    "dice": 1.0,\n    "nucleo_px": 361\n   },\n   "at_redondo": {\n    "dice": 0.9506,\n    "nucleo_px": 441\n   },\n   "dbnet_at": {\n    "dice": 0.9412,\n    "nucleo_px": 441\n   }\n  },\n  "2": {\n   "dbnet": {\n    "dice": 0.9407,\n    "nucleo_px": 705\n   },\n   "c1_rfijo": {\n    "dice": 0.9593,\n    "nucleo_px": 833\n   },\n   "c2_redondo": {\n    "dice": 0.9588,\n    "nucleo_px": 705\n   },\n   "c2": {\n    "dice": 1.0,\n    "nucleo_px": 705\n   },\n   "at_redondo": {\n    "dice": 0.9593,\n    "nucleo_px": 833\n   },\n   "dbnet_at": {\n    "dice": 0.9555,\n    "nucleo_px": 833\n   }\n  },\n  "4": {\n   "dbnet": {\n    "dice": 0.8231,\n    "nucleo_px": 1177\n   },\n   "c1_rfijo": {\n    "dice": 0.9132,\n    "nucleo_px": 1417\n   },\n   "c2_redondo": {\n    "dice": 0.9852,\n    "nucleo_px": 1177\n   },\n   "c2": {\n    "dice": 1.0,\n    "nucleo_px": 1177\n   },\n   "at_redondo": {\n    "dice": 0.991,\n    "nucleo_px": 1417\n   },\n   "dbnet_at": {\n    "dice": 0.9629,\n    "nucleo_px": 1417\n   }\n  },\n  "8": {\n   "dbnet": {\n    "dice": 0.7714,\n    "nucleo_px": 2245\n   },\n   "c1_rfijo": {\n    "dice": 0.8576,\n    "nucleo_px": 2750\n   },\n   "c2_redondo": {\n    "dice": 0.9663,\n    "nucleo_px": 2245\n   },\n   "c2": {\n    "dice": 0.9696,\n    "nucleo_px": 2245\n   },\n   "at_redondo": {\n    "dice": 0.9701,\n    "nucleo_px": 2750\n   },\n   "dbnet_at": {\n    "dice": 0.9419,\n    "nucleo_px": 2750\n   }\n  }\n }\n}'
CAMPANA = Path('/content/drive/MyDrive/tesis_imo_e3_secuencia_v4')

E3 = Path('/content/secuencia_e3_v4/e3')
sello = OUTPUT / 'firmas_secuencia.json'
if sello.exists() and json.loads(sello.read_text()) != FIRMAS:
    raise RuntimeError('La carpeta contiene otra versión; no mezclar')
for rel, texto in CONTENIDO.items():
    p = E3 / rel
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(texto, encoding='utf-8')
    assert sha_file(p) == FIRMAS[rel]
sello.write_text(json.dumps(FIRMAS, indent=2))
ref = E3.parent / 'resultados/geometria_sintetica.json'
ref.parent.mkdir(exist_ok=True)
ref.write_text(HISTORICO)
(OUTPUT / 'environment.freeze.txt').write_text(subprocess.check_output(
    [sys.executable, '-m', 'uv', 'pip', 'freeze', '--python', PY], text=True))
def verificar_original(semillas):
    for arm in ['dbnet', 'dbnet_at']:
        for seed in semillas:
            folder = CAMPANA / 'original' / f'unetat_{arm}_seed{seed}'
            r = json.loads((folder / 'result.json').read_text())
            assert r['completed'] and r['seed']==seed and r['variante']==arm
            assert sha_file(folder / r['best_file']) == r['best_sha256']
def entrenar(semillas):
    subprocess.run([PY, str(E3 / 'training/entrenar_original.py'), '--output', str(OUTPUT),
        '--variante', 'dbnet', 'dbnet_at', '--seeds', *map(str, semillas),
        '--voc', str(BASE), '--e2-scripts', str(PROJECT / 'scripts')], check=True)


In [ ]:
verificar_original([11,23,37,53,71,89,97,101,103,107])
for cfg in sorted((E3 / 'config').glob('hpo_original_t*.json')):
    subprocess.run([PY, str(E3 / 'training/entrenar_original.py'), '--output', str(OUTPUT / cfg.stem),
        '--variante', 'dbnet_at', '--seeds', '11', '--config', str(cfg),
        '--voc', str(BASE), '--e2-scripts', str(PROJECT / 'scripts')], check=True)
